# Indic-TTS `en+hi` → ONNX, with the lag diagnosed

One notebook, fresh session, nothing assumed. Downloads the checkpoints, loads
them, finds out why the speech lags, fixes it, exports to ONNX, quantises, and
measures.

**Run the cells in order.** Sections 4 and 5 are diagnostics that print things
you have to read before section 7 will produce anything worth shipping.

| Section | What it does | Stop and read? |
|---|---|---|
| 1 | Config — set `LANG` here and nowhere else | |
| 2 | Download repo + checkpoints (~1.4 GB) | |
| 3 | Install and load both stages | |
| 4 | **Sample rate audit** | **yes** |
| 5 | **Vocabulary audit** — what script this model accepts | **yes** |
| 6 | Numbers to words, English loanwords to Devanagari | |
| 7 | Duration map, speaking rate, silence trimming | **7a, 7b** |
| 7d | Romanised Hindi → Devanagari; pick your text format | **yes** |
| 8 | Export to ONNX with runtime speed control | |
| 9 | Static QDQ int8 + the ConvInteger gate | |
| 9b | Is the int8 damage real, or just duration drift? | **yes** |
| 9c | Re-quantise, excluding what int8 cannot take | |
| 9d | Calibration method sweep — the L1 fix | **yes** |
| 10a | Why long sentences break | **yes** |
| 10b | Chunked synthesis — the long-audio fix | |
| 10 | Honest latency measurement | |
| 11 | Listen and package | |
| 12 | **Playground** — type your own text and hear it | |

### Four things to know before you start

**Section 4 may end the investigation in ten seconds.** If the WAV sample rate
does not match what HiFi-GAN generated at, playback is stretched and that alone
sounds exactly like a lagging voice. Nothing else matters until it agrees.

**Section 5 decides what you can feed the model.** `en+hi` is advertised as
code-mixed, but if its vocabulary is Devanagari-only then Latin input is
silently discarded. Sections 6 and 7d handle both outcomes.

**Section 7d asks you to choose.** It plays the same sentence three ways — raw
Latin, dictionary-only, and fully transliterated — and you set `USE_V2` from
what you hear. Everything downstream, including quantisation calibration, then
uses that choice. Do not skip it: the calibration set must contain the same kind
of text the model will really receive.

**Section 9b exists because the obvious int8 metric lies.** Comparing two
waveforms sample-by-sample when they have different lengths measures alignment
drift, not damage. 9b asks the two questions separately, and 9c fixes whichever
one turns out to be real.

There are three hard gates. The variable-length test at the end of section 8
must show varying durations. The `ConvInteger = 0` check in 9 and 9c must pass
or the model will not load on Android. And RTF in section 10 must be under about
0.3 for chunked playback to stay ahead of the listener.

## 1. Configuration

Everything downstream reads from here. Change `LANG` and re-run the whole
notebook to do another language.

In [1]:
import os

LANG = "en+hi"     # "en+hi" | "hi" | "gu" | "or" | "en"

WORK_DIR  = "/kaggle/working"
ASSETS    = f"{WORK_DIR}/assets"
CKPT_ROOT = f"{ASSETS}/{LANG}"
REPO_DIR  = f"{WORK_DIR}/Indic-TTS"
ONNX_DIR  = f"{WORK_DIR}/onnx_{LANG}"
AUDIO_DIR = f"{WORK_DIR}/audio_{LANG}"

for d in (ASSETS, ONNX_DIR, AUDIO_DIR):
    os.makedirs(d, exist_ok=True)

FP_ONNX = f"{ONNX_DIR}/fastpitch_{LANG}.onnx"
HG_ONNX = f"{ONNX_DIR}/hifigan_{LANG}.onnx"
FP_Q    = FP_ONNX.replace(".onnx", ".static-qdq.onnx")
HG_Q    = HG_ONNX.replace(".onnx", ".static-qdq.onnx")

# Calibration sentences for static int8 quantisation. Static QDQ learns its
# activation ranges from exactly what you show it, so these have to look like
# what the app will really say: short instructions, places, numbers, names.
# Written in Devanagari; section 5 confirms whether that is the right script.
CALIB_TEXTS = [
    # short commands
    "मदद कीजिए, यहाँ आग लगी है।",
    "दो लोग घायल हो गए हैं।",
    "गोदाम नंबर तीन पर आइए।",
    "पानी और दवा भेजिए।",
    "रास्ता बंद है, दूसरे रास्ते से आइए।",
    "पाँच मिनट में पहुँच रहा हूँ।",
    "सब सुरक्षित हैं, चिंता मत कीजिए।",
    "एम्बुलेंस तुरंत भेजिए।",
    "बिजली काट दीजिए।",
    "यहाँ बहुत धुआँ है।",
    "छत पर बारह लोग फँसे हैं।",
    "पुल टूट गया है, गाड़ी मत लाइए।",
    "स्कूल के पास नाव भेजिए।",
    "रस्सी और टॉर्च चाहिए।",
    "मैं ठीक हूँ, आप कहाँ हैं?",
    # names and places - different phonetics from commands
    "लक्ष्मी, राजेश और अब्दुल यहाँ हैं।",
    "सेक्टर इक्कीस, ब्लॉक सी।",
    "नॉर्थ गेट से अंदर आइए।",
    # numbers, which appear constantly in real traffic
    "तीन सौ पैंतालीस लोग सुरक्षित हैं।",
    "रात आठ बजे तक पहुँचेंगे।",
    "दो किलोमीटर उत्तर की ओर।",
    # questions and exclamations - different prosody
    "क्या आप मुझे सुन सकते हैं?",
    "जल्दी कीजिए!",
    "कौन बोल रहा है?",
]

# Long sentences matter twice over: static quantisation learns its activation
# ranges from exactly what it is shown, so a calibrator that only saw four
# seconds has no idea how large activations get at twelve. That ceiling is
# both a quality problem and, past it, the reason long audio breaks.
CALIB_LONG = [
    "गोदाम नंबर तीन पे आग लगी है, दो लोग घायल हैं, एम्बुलेंस और दमकल दोनों तुरंत भेजिए, रास्ता नॉर्थ गेट से खुला है।",
    "सेक्टर इक्कीस में बारह लोग छत पर फँसे हैं, पानी बढ़ रहा है, नाव और रस्सी चाहिए, जल्दी कीजिए।",
    "पुल टूट गया है इसलिए गाड़ी मत लाइए, स्कूल के पीछे वाले रास्ते से आइए, वहाँ तीन सौ लोग सुरक्षित हैं।",
    "मैं ठीक हूँ, मेरे साथ पाँच लोग हैं, हम दो किलोमीटर उत्तर की ओर जा रहे हैं, रात आठ बजे तक पहुँच जाएँगे।",
]
CALIB_TEXTS = CALIB_TEXTS + CALIB_LONG

TEST_TEXT = "गोदाम नंबर तीन पे फायर लग गई है।"

# Raw copies. Section 6b re-normalises from these, so switching between the
# dictionary-only and transliterating versions stays idempotent instead of
# normalising already-normalised text.
CALIB_TEXTS_RAW = list(CALIB_TEXTS)
CALIB_LONG_RAW  = list(CALIB_LONG)
TEST_TEXT_RAW   = TEST_TEXT

print(f"LANG      {LANG}")
print(f"calib     {len(CALIB_TEXTS)} sentences ({len(CALIB_LONG)} long)")
print(f"ckpt      {CKPT_ROOT}")
print(f"onnx out  {ONNX_DIR}")

LANG      en+hi
calib     28 sentences (4 long)
ckpt      /kaggle/working/assets/en+hi
onnx out  /kaggle/working/onnx_en+hi


## 2. Download

Roughly 1.4 GB. Safe to re-run — anything already present is skipped, and an
interrupted download resumes rather than restarting.

In [2]:
import subprocess, sys, zipfile

REL = ("https://github.com/AI4Bharat/Indic-TTS/releases/download/"
       "v1-checkpoints-release")

def sh(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().split("\n")[-12:]
    print("\n".join(tail))
    if check and r.returncode != 0:
        raise RuntimeError(f"failed ({r.returncode}): {' '.join(cmd)}")
    return r.returncode

# ---- the repo (AI4Bharat's Coqui TTS fork) ----
if not os.path.isdir(REPO_DIR):
    sh(["git", "clone", "-q", "--depth", "1",
        "https://github.com/AI4Bharat/Indic-TTS.git", REPO_DIR])
else:
    print("repo already present")

# ---- the checkpoint archive ----
zpath = f"{ASSETS}/{LANG}.zip"
if os.path.isdir(CKPT_ROOT) and os.listdir(CKPT_ROOT):
    print(f"{LANG} already unpacked at {CKPT_ROOT}")
else:
    have = os.path.getsize(zpath) if os.path.exists(zpath) else 0
    if have < 1_000_000_000:
        print(f"downloading {LANG}.zip (~1.4 GB)"
              f"{' - resuming' if have else ''} ...")
        # "+" is literal in a URL path (it only means space in a query string),
        # so en+hi.zip works as-is. Encoding it costs nothing and removes doubt.
        asset = LANG.replace("+", "%2B") + ".zip"
        rc = sh(["curl", "-L", "-f", "--retry", "3", "-C", "-",
                 "-o", zpath, f"{REL}/{asset}"], check=False)
        if rc != 0 and "+" in LANG:
            print("retrying with the literal '+' ...")
            rc = sh(["curl", "-L", "-f", "--retry", "3", "-C", "-",
                     "-o", zpath, f"{REL}/{LANG}.zip"], check=False)
        if rc != 0:
            raise RuntimeError(
                f"{LANG}.zip could not be downloaded. Open\n"
                f"  {REL.rsplit('/download/', 1)[0]}/tag/v1-checkpoints-release\n"
                f"and confirm an asset with that exact name exists.")
    print("unpacking ...")
    with zipfile.ZipFile(zpath) as z:
        z.extractall(ASSETS)
    print("done")

# ---- show what we got; the layout matters for section 3 ----
print("\nlayout:")
for root, dirs, files in os.walk(CKPT_ROOT):
    depth = root.replace(CKPT_ROOT, "").count(os.sep)
    if depth > 2:
        continue
    print("  " * depth + os.path.basename(root) + "/")
    for f in sorted(files)[:8]:
        mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print("  " * (depth + 1) + f"{f}  ({mb:.1f} MB)")

repo already present
en+hi already unpacked at /kaggle/working/assets/en+hi

layout:
en+hi/
  fastpitch/
    best_model.pth  (651.4 MB)
    config.json  (0.0 MB)
    config_patched.json  (0.0 MB)
    speakers.pth  (0.0 MB)
  hifigan/
    best_model.pth  (1016.4 MB)
    config.json  (0.0 MB)
    config_patched.json  (0.0 MB)


## 3. Install and load

Indic-TTS is a fork of Coqui TTS from 2022. Its pinned dependency list fights a
2026 environment, so we install `coqui-tts` (the maintained fork, imports as
`TTS`) rather than the abandoned original.

`pillow<12` is pinned deliberately: Pillow 12 moved `_Ink` out of `PIL._typing`
and current torchvision builds still expect the 11.x layout. Without the pin you
get an import error that names none of those three things.

**The config patch matters.** The released `config.json` files carry absolute
paths from AI4Bharat's own training machine. Coqui follows them literally, so
loading fails on any other computer until they are rewritten.

In [3]:
import sys, subprocess

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-1500:])
    return r.returncode

try:
    import TTS
    print("TTS already importable")
except ImportError:
    print("installing coqui-tts ...")
    pip("coqui-tts", "pillow<12")

print("installing onnx tooling ...")
pip("onnx", "onnxruntime")

import torch, numpy as np, soundfile as sf, librosa, json, glob, time
import onnx, onnxruntime as ort
from TTS.config import load_config
from TTS.tts.models import setup_model as setup_tts_model
from TTS.vocoder.models import setup_model as setup_vocoder_model

print(f"\ntorch {torch.__version__} | onnxruntime {ort.__version__}")
DEVICE = "cpu"   # export on CPU; tracing on GPU bakes in device-specific ops

TTS already importable
installing onnx tooling ...

torch 2.10.0+cu128 | onnxruntime 1.29.0


In [4]:
# ---- patch the absolute paths, then load both stages ------------------
RELOCATE = {"speakers_file", "d_vector_file", "speaker_file",
            "language_ids_file", "stats_path"}
BLANK    = {"phoneme_cache_path", "output_path", "datasets",
            "test_sentences_file", "restore_path", "run_name_dir"}

def patch(node, search_dir, trail=""):
    """Rewrite file references to where the file actually is now; blank the
    fields that only mattered during training."""
    if isinstance(node, dict):
        for k in list(node.keys()):
            v = node[k]
            if k in BLANK and v:
                node[k] = [] if isinstance(v, list) else None
            elif k in RELOCATE and isinstance(v, str) and v:
                if os.path.exists(v):
                    continue
                hits = glob.glob(f"{search_dir}/**/{os.path.basename(v)}",
                                 recursive=True)
                node[k] = hits[0] if hits else None
                print(f"    {trail}{k}: -> {node[k]}")
            else:
                patch(v, search_dir, trail + k + ".")
    elif isinstance(node, list):
        for v in node:
            patch(v, search_dir, trail)

def load_stage(stage, setup_fn):
    d = os.path.join(CKPT_ROOT, stage)
    ck = glob.glob(f"{d}/**/best_model.pth", recursive=True)
    cf = glob.glob(f"{d}/**/config.json", recursive=True)
    cf = [p for p in cf if "patched" not in p]
    if not ck or not cf:
        raise FileNotFoundError(f"{stage}: need best_model.pth and config.json "
                                f"under {d}, found {len(ck)} / {len(cf)}")
    print(f"  {stage}: {os.path.basename(ck[0])} + {os.path.basename(cf[0])}")
    raw = json.load(open(cf[0]))
    patch(raw, d)
    fixed = os.path.join(d, "config_patched.json")
    json.dump(raw, open(fixed, "w"))
    cfg = load_config(fixed)
    m = setup_fn(cfg)
    m.load_checkpoint(cfg, ck[0], eval=True)
    return m.eval(), cfg

print("loading fastpitch ...")
tts_model, tts_config = load_stage("fastpitch", setup_tts_model)
print("loading hifigan ...")
gen, gen_config = load_stage("hifigan", setup_vocoder_model)

tokenizer = tts_model.tokenizer

# ---- speaker ----
SPEAKER_ID, SPEAKER_NAME = 0, None
try:
    n2i = tts_model.speaker_manager.name_to_id
    names = list(n2i.keys())
    SPEAKER_NAME, SPEAKER_ID = names[0], n2i[names[0]]
    print(f"\nspeakers ({len(names)}): {names[:8]}"
          f"{' ...' if len(names) > 8 else ''}")
    print(f"using: {SPEAKER_NAME} (id {SPEAKER_ID})")
except Exception as e:
    print(f"\nsingle-speaker model or no speaker manager ({e})")

n_params = sum(p.numel() for p in tts_model.parameters())
v_params = sum(p.numel() for p in gen.parameters())
print(f"\nfastpitch {n_params/1e6:.1f}M   hifigan {v_params/1e6:.1f}M")

loading fastpitch ...
  fastpitch: best_model.pth + config.json
    model_args.speakers_file: -> /kaggle/working/assets/en+hi/fastpitch/speakers.pth
    speakers_file: -> /kaggle/working/assets/en+hi/fastpitch/speakers.pth
loading hifigan ...
  hifigan: best_model.pth + config.json

speakers (4): ['female', 'hindifemale', 'hindimale', 'male']
using: female (id 0)

fastpitch 53.4M   hifigan 13.9M


## 4. Sample rate audit — read this before anything else

FastPitch produces mel spectrograms. **HiFi-GAN turns them into samples, at its
own trained rate.** That vocoder rate is the only one that matters for playback.

If you write the WAV claiming a different rate, the audio is stretched or
compressed. 22050 written as 16000 plays 1.38× too slow: the voice drags and
drops in pitch. That is exactly what "lag when speaking the words" sounds like.

The cell also cross-checks empirically, because a config can be wrong too.

In [5]:
def cfg_sr(cfg):
    """Find sample_rate wherever this Coqui config keeps it."""
    for path in (("audio", "output_sample_rate"), ("audio", "sample_rate"),
                 ("output_sample_rate",), ("sample_rate",)):
        node = cfg
        try:
            for k in path:
                node = node[k] if isinstance(node, dict) else getattr(node, k)
            if node:
                return int(node), ".".join(path)
        except Exception:
            continue
    return None, None

fp_sr, fp_where = cfg_sr(tts_config)
hg_sr, hg_where = cfg_sr(gen_config)

print("=" * 64)
print("SAMPLE RATES")
print("=" * 64)
print(f"  fastpitch : {fp_sr}   (from {fp_where})")
print(f"  hifigan   : {hg_sr}   (from {hg_where})")

sr = hg_sr or fp_sr or 22050
print(f"\n  -> playback rate is {sr} Hz (the vocoder's)")

if fp_sr and hg_sr and fp_sr != hg_sr:
    print(f"\n  *** the two stages disagree ({fp_sr} vs {hg_sr}) ***")
    print(f"  Use {hg_sr}. Anything else stretches the audio.")

# hop length, needed later to convert durations to milliseconds
HOP = None
for k in ("hop_length", "hop_size"):
    try:
        HOP = int(tts_config.audio[k]); break
    except Exception:
        pass
HOP = HOP or 256
N_MELS = tts_config.audio.get("num_mels", 80)
print(f"  hop_length = {HOP}   num_mels = {N_MELS}")

# ---- empirical cross-check ----
x = torch.LongTensor(tokenizer.text_to_ids(TEST_TEXT)).unsqueeze(0)
spk = torch.LongTensor([SPEAKER_ID])
with torch.no_grad():
    out = tts_model.inference(x, aux_input={"d_vectors": None,
                                            "speaker_ids": spk})
    mel = out["model_outputs"]
    NEEDS_TRANSPOSE = mel.shape[-1] == N_MELS   # (B,T,C) -> (B,C,T)
    mel_v = mel.transpose(1, 2) if NEEDS_TRANSPOSE else mel
    wav = np.asarray(gen(mel_v)).squeeze()

print(f"\n  text          : {TEST_TEXT}")
print(f"  mel           : {tuple(mel.shape)}  NEEDS_TRANSPOSE={NEEDS_TRANSPOSE}")
print(f"  samples       : {len(wav)}")
print(f"  samples/frame : {len(wav)/mel_v.shape[-1]:.0f}  (should equal hop "
      f"{HOP})")
print(f"  duration      : {len(wav)/sr:.2f} s at {sr} Hz")
print("\n  Say that sentence out loud and time yourself. If the number above")
print("  is much larger, the rate is still wrong and that IS the lag.")

sf.write(f"{AUDIO_DIR}/00_rate_check.wav", wav.astype(np.float32), sr)
from IPython.display import Audio, display
display(Audio(f"{AUDIO_DIR}/00_rate_check.wav"))

गोदाम नंबर तीन पे फायर लग गई है।
Character '।' not found in the vocabulary. Discarding it.


SAMPLE RATES
  fastpitch : 22050   (from audio.sample_rate)
  hifigan   : 22050   (from audio.sample_rate)

  -> playback rate is 22050 Hz (the vocoder's)
  hop_length = 256   num_mels = 80

  text          : गोदाम नंबर तीन पे फायर लग गई है।
  mel           : (1, 206, 80)  NEEDS_TRANSPOSE=True
  samples       : 52736
  samples/frame : 256  (should equal hop 256)
  duration      : 2.39 s at 22050 Hz

  Say that sentence out loud and time yourself. If the number above
  is much larger, the rate is still wrong and that IS the lag.


## 5. Vocabulary audit — what script does this model accept?

`en+hi` is advertised as code-mixed. That does not guarantee its character
vocabulary contains Latin letters — the training text may have been
transliterated to Devanagari before tokenisation.

This matters enormously. Characters outside the vocabulary do not vanish
politely: depending on the tokenizer they collapse into pad or blank ids, and
FastPitch then **predicts a duration for those ids**. You get silence where a
word should have been, which reads as lag between words.

Read the survival rates below. Whichever row keeps most of its characters is the
script you must write in.

In [6]:
chars = tokenizer.characters
VOCAB = getattr(chars, "vocab", None) or getattr(chars, "_vocab", None) or []
VOCAB_SET = set(VOCAB)

print("=" * 64)
print("VOCABULARY")
print("=" * 64)
print(f"  size: {chars.num_chars}")
print(f"  set : {''.join(VOCAB)}")

def block(lo, hi):
    return sum(1 for c in VOCAB if len(c) == 1 and lo <= ord(c) <= hi)

print("\n  script coverage:")
for name, lo, hi in [
    ("latin lower", 0x0061, 0x007A), ("latin upper", 0x0041, 0x005A),
    ("devanagari ", 0x0900, 0x097F), ("bengali    ", 0x0980, 0x09FF),
    ("gujarati   ", 0x0A80, 0x0AFF), ("odia       ", 0x0B00, 0x0B7F),
    ("tamil      ", 0x0B80, 0x0BFF), ("telugu     ", 0x0C00, 0x0C7F),
    ("digits     ", 0x0030, 0x0039),
]:
    n = block(lo, hi)
    print(f"    {name}  {n:3d}  {'YES' if n > 5 else 'no'}")

print("\n  what survives tokenisation:")
for label, t in [
    ("latin hinglish ", "Godown number teen pe fire lag gayi hai."),
    ("devanagari hi  ", "गोदाम नंबर तीन पर आइए।"),
    ("deva hinglish  ", "गोदाम नंबर तीन पे फायर लग गई है।"),
    ("UPPER latin    ", "GODOWN NUMBER THREE."),
    ("digits         ", "सेक्टर 21 में 12 लोग।"),
]:
    kept = "".join(c for c in t if c in VOCAB_SET)
    ids = tokenizer.text_to_ids(t)
    pct = 100 * len(kept) / max(len(t), 1)
    print(f"    {label}: {len(ids):3d} ids from {len(t):3d} chars "
          f"({pct:4.0f}% kept)  '{kept[:40]}'")

HAS_LATIN = block(0x0061, 0x007A) > 5
HAS_DIGITS = block(0x0030, 0x0039) > 5
print(f"""
  -> latin in vocabulary : {HAS_LATIN}
  -> digits in vocabulary: {HAS_DIGITS}

  If latin is False, section 6's transliteration is mandatory, not optional.
  If digits is False, numbers MUST be spelled out as words. They usually are
  not in vocabulary, and unhandled digits are a classic source of gaps.
""")

VOCABULARY
  size: 137
  set : <PAD>!¡'(),-.:;¿?  !%,-.01234568:;?`abcdefghijklmnopqrstuvwxyz ँंःअआइईउऊऋएऐऑओऔकखगघङचछजझञटठडढणतथदधनपफबभमयरऱलळवशषसह़ािीुूृॅेैॉोौ्क़ख़ग़ज़ड़ढ़फ़ॠ‍‎–—…<BLNK>

  script coverage:
    latin lower   26  YES
    latin upper    0  no
    devanagari    72  YES
    bengali        0  no
    gujarati       0  no
    odia           0  no
    tamil          0  no
    telugu         0  no
    digits         8  YES

  what survives tokenisation:
    latin hinglish :  40 ids from  40 chars (  98% kept)  'odown number teen pe fire lag gayi hai.'
    devanagari hi  :  21 ids from  22 chars (  95% kept)  'गोदाम नंबर तीन पर आइए'
    deva hinglish  :  31 ids from  32 chars (  97% kept)  'गोदाम नंबर तीन पे फायर लग गई है'
    UPPER latin    :  20 ids from  20 chars (  15% kept)  '  .'
    digits         :  20 ids from  21 chars (  95% kept)  'सेक्टर 21 में 12 लोग'

  -> latin in vocabulary : True
  -> digits in vocabulary: True

  If latin is False, section 6's transliteration is mandat

## 6. Text normalisation

Two rules, both cheap, both mandatory:

1. **Never send a character the vocabulary lacks.** Transliterate or drop it
   deliberately, rather than letting the tokenizer mangle it silently.
2. **Numbers become words.** `23` is not something FastPitch was trained to
   pronounce. Every emergency message has numbers in it — counts, sectors,
   distances, times — so this is the highest-value piece of TTS plumbing you
   will write.

`LATIN_TO_DEVA` starts with about forty words a rescue worker actually says.
Grow it from real transcripts; it is a dictionary, not an algorithm.

In [7]:
import re, unicodedata

# ---- Hindi numbers. Irregular all the way to 99, hence the table. -----
_HI_ONES = ["शून्य", "एक", "दो", "तीन", "चार", "पाँच", "छह", "सात", "आठ", "नौ"]
_HI_10_99 = {
    10:"दस",11:"ग्यारह",12:"बारह",13:"तेरह",14:"चौदह",15:"पंद्रह",16:"सोलह",
    17:"सत्रह",18:"अठारह",19:"उन्नीस",20:"बीस",21:"इक्कीस",22:"बाईस",
    23:"तेईस",24:"चौबीस",25:"पच्चीस",26:"छब्बीस",27:"सत्ताईस",28:"अट्ठाईस",
    29:"उनतीस",30:"तीस",31:"इकतीस",32:"बत्तीस",33:"तैंतीस",34:"चौंतीस",
    35:"पैंतीस",36:"छत्तीस",37:"सैंतीस",38:"अड़तीस",39:"उनतालीस",40:"चालीस",
    41:"इकतालीस",42:"बयालीस",43:"तैंतालीस",44:"चवालीस",45:"पैंतालीस",
    46:"छियालीस",47:"सैंतालीस",48:"अड़तालीस",49:"उनचास",50:"पचास",
    51:"इक्यावन",52:"बावन",53:"तिरपन",54:"चौवन",55:"पचपन",56:"छप्पन",
    57:"सत्तावन",58:"अट्ठावन",59:"उनसठ",60:"साठ",61:"इकसठ",62:"बासठ",
    63:"तिरसठ",64:"चौंसठ",65:"पैंसठ",66:"छियासठ",67:"सड़सठ",68:"अड़सठ",
    69:"उनहत्तर",70:"सत्तर",71:"इकहत्तर",72:"बहत्तर",73:"तिहत्तर",
    74:"चौहत्तर",75:"पचहत्तर",76:"छिहत्तर",77:"सतहत्तर",78:"अठहत्तर",
    79:"उन्यासी",80:"अस्सी",81:"इक्यासी",82:"बयासी",83:"तिरासी",
    84:"चौरासी",85:"पचासी",86:"छियासी",87:"सत्तासी",88:"अट्ठासी",
    89:"नवासी",90:"नब्बे",91:"इक्यानवे",92:"बानवे",93:"तिरानवे",
    94:"चौरानवे",95:"पंचानवे",96:"छियानवे",97:"सत्तानवे",98:"अट्ठानवे",
    99:"निन्यानवे",
}

def hi_num(n):
    if n < 0:
        return "माइनस " + hi_num(-n)
    if n < 10:
        return _HI_ONES[n]
    if n < 100:
        return _HI_10_99[n]
    if n < 1000:
        h, r = divmod(n, 100)
        return _HI_ONES[h] + " सौ" + (" " + hi_num(r) if r else "")
    if n < 100000:
        t, r = divmod(n, 1000)
        return hi_num(t) + " हज़ार" + (" " + hi_num(r) if r else "")
    if n < 10000000:
        l, r = divmod(n, 100000)
        return hi_num(l) + " लाख" + (" " + hi_num(r) if r else "")
    c, r = divmod(n, 10000000)
    return hi_num(c) + " करोड़" + (" " + hi_num(r) if r else "")

# ---- Latin -> Devanagari for the words users actually say --------------
LATIN_TO_DEVA = {
    "godown":"गोदाम","fire":"फायर","number":"नंबर","sector":"सेक्टर",
    "block":"ब्लॉक","gate":"गेट","road":"रोड","police":"पुलिस",
    "ambulance":"एम्बुलेंस","hospital":"अस्पताल","help":"हेल्प",
    "emergency":"इमरजेंसी","rescue":"रेस्क्यू","team":"टीम","water":"वाटर",
    "medicine":"दवा","building":"बिल्डिंग","floor":"फ्लोर","unit":"यूनिट",
    "over":"ओवर","check":"चेक","ok":"ओके","okay":"ओके","yes":"हाँ",
    "no":"नहीं","stop":"स्टॉप","start":"स्टार्ट","kilometre":"किलोमीटर",
    "kilometer":"किलोमीटर","km":"किलोमीटर","metre":"मीटर","meter":"मीटर",
    "minute":"मिनट","hour":"घंटा","north":"नॉर्थ","south":"साउथ",
    "east":"ईस्ट","west":"वेस्ट","left":"लेफ्ट","right":"राइट",
    "flood":"बाढ़","bridge":"पुल","school":"स्कूल","village":"गाँव",
    "boat":"नाव","rope":"रस्सी","light":"लाइट","radio":"रेडियो",
}

def normalise(text, verbose=False):
    """NFC -> numbers to words -> transliterate latin -> drop the rest."""
    t = unicodedata.normalize("NFC", text)

    # numbers first, before letter substitution can touch them
    t = re.sub(r"\d+", lambda m: " " + hi_num(int(m.group())) + " ", t)

    # longest match first, so "kilometre" wins over "km"
    for w in sorted(LATIN_TO_DEVA, key=len, reverse=True):
        t = re.sub(rf"\b{re.escape(w)}\b", LATIN_TO_DEVA[w], t,
                   flags=re.IGNORECASE)

    if not HAS_LATIN:
        leftover = set(re.findall(r"[A-Za-z]+", t))
        if leftover and verbose:
            print(f"    no mapping, dropped: {sorted(leftover)}")
        t = re.sub(r"[A-Za-z]+", "", t)

    if VOCAB_SET:
        dropped = {c for c in t if c not in VOCAB_SET and not c.isspace()}
        if dropped and verbose:
            print(f"    not in vocab, dropped: {sorted(dropped)}")
        t = "".join(c for c in t if c in VOCAB_SET or c.isspace())

    return re.sub(r"\s+", " ", t).strip()

print("=" * 64)
print("NORMALISATION")
print("=" * 64)
for s in [
    "Godown number 3 pe fire lag gayi hai.",
    "गोदाम नंबर 3 पे फायर लग गई है।",
    "Sector 21 mein 12 log fase hain, ambulance bhejo.",
    "5 minute mein pahunch raha hoon.",
    "मदद कीजिए, यहाँ आग लगी है।",
]:
    print(f"\n  in  : {s}")
    out = normalise(s, verbose=True)
    print(f"  out : {out}")
    print(f"  ids : {len(tokenizer.text_to_ids(s))} -> "
          f"{len(tokenizer.text_to_ids(out))}")

# normalise the calibration set too, so quantisation sees real input
CALIB_TEXTS = [normalise(t) for t in CALIB_TEXTS]
TEST_TEXT = normalise(TEST_TEXT)
print(f"\n  TEST_TEXT -> {TEST_TEXT}")

NORMALISATION

  in  : Godown number 3 pe fire lag gayi hai.
  out : गोदाम नंबर तीन pe फायर lag gayi hai.
  ids : 37 -> 36

  in  : गोदाम नंबर 3 पे फायर लग गई है।
    not in vocab, dropped: ['।']
  out : गोदाम नंबर तीन पे फायर लग गई है
  ids : 29 -> 31

  in  : Sector 21 mein 12 log fase hain, ambulance bhejo.
  out : सेक्टर इक्कीस mein बारह log fase hain, एम्बुलेंस bhejo.
  ids : 49 -> 55

  in  : 5 minute mein pahunch raha hoon.
  out : पाँच मिनट mein pahunch raha hoon.
  ids : 32 -> 33

  in  : मदद कीजिए, यहाँ आग लगी है।
    not in vocab, dropped: ['।']
  out : मदद कीजिए, यहाँ आग लगी है
  ids : 25 -> 25

  TEST_TEXT -> गोदाम नंबर तीन पे फायर लग गई है


## 7. Where the time goes, and how to take it back

Three things in order:

**7a — the duration map.** FastPitch predicts frames per token. This prints them
with a bar chart and a breakdown by symbol class. If spaces, punctuation and
unknown ids are more than about 20% of total frames, that is your inter-word lag.

**7b — `length_scale`.** Multiplies every duration. Below 1.0 speaks faster.
0.85–1.0 is safe on FastPitch; below 0.8 starts to sound clipped.

**7c — silence trimming.** Removes leading and trailing padding and shortens long
internal gaps. Worth doing regardless: for a walkie-talkie, every 100 ms of dead
air at the front is 100 ms the user feels.

In [8]:
# ---- 7a: per-token durations ------------------------------------------
def duration_map(text, length_scale=None, show_tokens=True):
    m = tts_model
    ids = tokenizer.text_to_ids(text)
    x = torch.LongTensor(ids).unsqueeze(0)
    spk = torch.LongTensor([SPEAKER_ID])

    old = getattr(m, "length_scale", 1.0)
    if length_scale is not None:
        m.length_scale = length_scale
    try:
        with torch.no_grad():
            x_mask = torch.ones_like(x).unsqueeze(1).float()
            o_en, x_mask, g, _ = m._forward_encoder(x, x_mask, spk)
            o_dr_log = m.duration_predictor(o_en.squeeze(), x_mask)
            o_dr = m.format_durations(o_dr_log, x_mask).squeeze(1)
    finally:
        m.length_scale = old

    durs = o_dr[0].cpu().numpy()

    def sym(i):
        try:
            c = VOCAB[i]
        except Exception:
            return f"<{i}>"
        return {" ": "SP", "": "<empty>"}.get(c, c)

    total = float(durs.sum())
    print(f"text   : {text}")
    print(f"tokens : {len(ids)}   frames: {total:.0f}   "
          f"audio: {total*HOP/sr:.2f} s")

    if show_tokens:
        print(f"\n{'idx':>4} {'id':>5} {'char':>8} {'frames':>7} {'ms':>6}  bar")
        print("-" * 60)
        for i, (tid, d) in enumerate(zip(ids, durs)):
            ms = d * HOP / sr * 1000
            flag = "  <-- LONG" if ms > 250 else ""
            print(f"{i:4d} {tid:5d} {sym(tid):>8} {d:7.1f} {ms:6.0f}"
                  f"  {'#' * min(int(d), 34)}{flag}")

    import collections
    buckets = collections.defaultdict(float)
    for tid, d in zip(ids, durs):
        c = sym(tid)
        if c == "SP":
            buckets["space"] += d
        elif c in ".,!?।;:-":
            buckets["punctuation"] += d
        elif c.startswith("<"):
            buckets["UNKNOWN/pad"] += d
        else:
            buckets["speech"] += d

    print("\nframes by class:")
    for k, v in sorted(buckets.items(), key=lambda kv: -kv[1]):
        print(f"   {k:14s} {v:7.0f}  ({100*v/total:5.1f}%)  {v*HOP/sr:5.2f} s")

    nonspeech = 100 * (total - buckets['speech']) / total
    print(f"\n   non-speech = {nonspeech:.1f}%")
    if nonspeech > 20:
        print("   -> THAT is your inter-word lag. Fix the text, not the model.")
    else:
        print("   -> mostly speech. If it still drags, use length_scale (7b).")
    return ids, durs

_ = duration_map(TEST_TEXT)

text   : गोदाम नंबर तीन पे फायर लग गई है
tokens : 31   frames: 206   audio: 2.39 s

 idx    id     char  frames     ms  bar
------------------------------------------------------------
   0    76        ग     9.0    104  #########
   1   120        ो    12.0    139  ############
   2    91        द     4.0     46  ####
   3   110        ा    11.0    128  ###########
   4    98        म     5.0     58  #####
   5    14       SP     8.0     93  ########
   6    93        न     4.0     46  ####
   7    60        ं     6.0     70  ######
   8    96        ब     6.0     70  ######
   9   100        र    10.0    116  ##########
  10    14       SP    11.0    128  ###########
  11    89        त     2.0     23  ##
  12   112        ी     7.0     81  #######
  13    93        न     9.0    104  #########
  14    14       SP     7.0     81  #######
  15    94        प     2.0     23  ##
  16   117        े     7.0     81  #######
  17    14       SP     9.0    104  #########
  18    95        फ 

In [9]:
# ---- 7b: length_scale sweep -------------------------------------------
from IPython.display import Audio, display

def synth(text, length_scale=1.0, do_normalise=True):
    """PyTorch path, so speed can vary without re-exporting."""
    m = tts_model
    t = normalise(text) if do_normalise else text
    x = torch.LongTensor(tokenizer.text_to_ids(t)).unsqueeze(0)
    spk = torch.LongTensor([SPEAKER_ID])
    old = getattr(m, "length_scale", 1.0)
    m.length_scale = length_scale
    try:
        with torch.no_grad():
            o = m.inference(x, aux_input={"d_vectors": None,
                                          "speaker_ids": spk})
            mel = o["model_outputs"]
            mel = mel.transpose(1, 2) if NEEDS_TRANSPOSE else mel
            w = gen(mel)
    finally:
        m.length_scale = old
    return np.asarray(w).squeeze()

print(f"text: {TEST_TEXT}\n")
for ls in [1.0, 0.95, 0.9, 0.85, 0.8]:
    w = synth(TEST_TEXT, length_scale=ls, do_normalise=False)
    p = f"{AUDIO_DIR}/ls_{ls}.wav"
    sf.write(p, w.astype(np.float32), sr)
    print(f"  length_scale {ls:.2f}  ->  {len(w)/sr:.2f} s")
    display(Audio(p))

print("""
Pick the fastest one that still sounds natural. That value is what you will
pass at run time in section 8 — it becomes a graph input, not a baked constant,
so alerts can be slowed for intelligibility while normal traffic stays quick.
""")
LENGTH_SCALE = 0.9   # <-- set this to whatever you picked

text: गोदाम नंबर तीन पे फायर लग गई है

  length_scale 1.00  ->  2.39 s


  length_scale 0.95  ->  2.26 s


  length_scale 0.90  ->  2.19 s


  length_scale 0.85  ->  2.02 s


  length_scale 0.80  ->  1.93 s



Pick the fastest one that still sounds natural. That value is what you will
pass at run time in section 8 — it becomes a graph input, not a baked constant,
so alerts can be slowed for intelligibility while normal traffic stays quick.



In [10]:
# ---- 7c: trim silence, collapse gaps ----------------------------------
def trim_and_collapse(wav, sr, top_db=35, max_gap_ms=180, edge_pad_ms=20):
    """Strip edge silence; shorten internal pauses longer than max_gap_ms."""
    wav = np.asarray(wav, dtype=np.float32)
    trimmed, _ = librosa.effects.trim(wav, top_db=top_db)
    pad = int(edge_pad_ms * sr / 1000)
    out = np.concatenate([np.zeros(pad, np.float32), trimmed,
                          np.zeros(pad, np.float32)])

    iv = librosa.effects.split(out, top_db=top_db)
    if len(iv) <= 1:
        return out
    max_gap = int(max_gap_ms * sr / 1000)
    pieces = [out[iv[0][0]:iv[0][1]]]
    for (_, prev_end), (start, end) in zip(iv[:-1], iv[1:]):
        pieces.append(out[prev_end:prev_end + min(start - prev_end, max_gap)])
        pieces.append(out[start:end])
    return np.concatenate(pieces)

raw = synth(TEST_TEXT, length_scale=LENGTH_SCALE, do_normalise=False)
fix = trim_and_collapse(raw, sr)
print(f"  before {len(raw)/sr:.2f} s   after {len(fix)/sr:.2f} s   "
      f"({100*(1-len(fix)/len(raw)):.0f}% shorter)")
for nm, w in [("raw", raw), ("trimmed", fix)]:
    p = f"{AUDIO_DIR}/{nm}.wav"
    sf.write(p, w.astype(np.float32), sr)
    print(f"\n{nm}")
    display(Audio(p))
print("""
If trimming eats the first or last phoneme, raise edge_pad_ms.
If it still sounds gappy, lower max_gap_ms toward 120.
""")

  before 2.19 s   after 2.23 s   (-2% shorter)

raw



trimmed



If trimming eats the first or last phoneme, raise edge_pad_ms.
If it still sounds gappy, lower max_gap_ms toward 120.



## 7d. Romanised Hindi — the transliteration your dictionary cannot do

A dictionary handles English loanwords: `sector` must become सेक्टर, not the
सेक्टोर a transliterator would produce. But romanised **Hindi** — `bhejo`,
`hain`, `fase`, `mein`, `turant` — is not a finite list you can look up. Those
are Hindi words spelled in Latin letters, and there are as many of them as there
are Hindi words.

Leaving them alone produces the worst possible input:

```
सेक्टर इक्कीस mein बारह log fase hain, एम्बुलेंस turant bhejo.
```

Devanagari and Latin alternating mid-sentence. So: curated dictionary first for
loanwords, then transliterate whatever Latin survives.

**Listen before adopting.** ITRANS transliteration of casual romanisation is
imperfect — `hain` may come out as हैन rather than हैं. But a consistently
slightly-wrong Devanagari word usually sounds better than a Latin word the model
has to guess at. The cell plays all three and lets you choose.

In [11]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "indic-transliteration"], capture_output=True)

try:
    from indic_transliteration import sanscript
    from indic_transliteration.sanscript import transliterate
    HAVE_TRANSLIT = True
except ImportError as e:
    print(f"indic-transliteration unavailable ({e}); v2 will fall back to v1")
    HAVE_TRANSLIT = False

def to_deva(word):
    """ITRANS is the scheme closest to how Indians actually type Hindi."""
    if not HAVE_TRANSLIT:
        return word
    try:
        return transliterate(word.lower(), sanscript.ITRANS,
                             sanscript.DEVANAGARI)
    except Exception:
        return word

def normalise_v2(text, verbose=False):
    """Dictionary for loanwords, then transliterate the romanised Hindi."""
    t = unicodedata.normalize("NFC", text)
    t = re.sub(r"\d+", lambda m: " " + hi_num(int(m.group())) + " ", t)

    # 1. curated English loanwords, longest match first
    for w in sorted(LATIN_TO_DEVA, key=len, reverse=True):
        t = re.sub(rf"\b{re.escape(w)}\b", LATIN_TO_DEVA[w], t,
                   flags=re.IGNORECASE)

    # 2. whatever Latin remains is romanised Hindi
    leftover = re.findall(r"[A-Za-z]+", t)
    if leftover and verbose:
        print(f"    transliterating: {leftover}")
    t = re.sub(r"[A-Za-z]+", lambda m: to_deva(m.group()), t)

    if VOCAB_SET:
        t = "".join(c for c in t if c in VOCAB_SET or c.isspace())
    return re.sub(r"\s+", " ", t).strip()


TRICKY = [
    "Sector 21 mein 12 log fase hain, ambulance turant bhejo.",
    "Godown number 3 pe fire lag gayi hai.",
    "5 minute mein pahunch raha hoon, tension mat lo.",
]

print("=" * 70)
print("v1 (dictionary only)  vs  v2 (dictionary + transliteration)")
print("=" * 70)
for s_ in TRICKY:
    print(f"\n  in : {s_}")
    print(f"  v1 : {normalise(s_)}")
    print(f"  v2 : {normalise_v2(s_, verbose=True)}")

v1 (dictionary only)  vs  v2 (dictionary + transliteration)

  in : Sector 21 mein 12 log fase hain, ambulance turant bhejo.
  v1 : सेक्टर इक्कीस mein बारह log fase hain, एम्बुलेंस turant bhejo.
    transliterating: ['mein', 'log', 'fase', 'hain', 'turant', 'bhejo']
  v2 : सेक्टर इक्कीस मेइन् बारह लोग् फ़से हैन्, एम्बुलेंस तुरन्त् भेजो.

  in : Godown number 3 pe fire lag gayi hai.
  v1 : गोदाम नंबर तीन pe फायर lag gayi hai.
    transliterating: ['pe', 'lag', 'gayi', 'hai']
  v2 : गोदाम नंबर तीन पे फायर लग् गयि है.

  in : 5 minute mein pahunch raha hoon, tension mat lo.
  v1 : पाँच मिनट mein pahunch raha hoon, tension mat lo.
    transliterating: ['mein', 'pahunch', 'raha', 'hoon', 'tension', 'mat', 'lo']
  v2 : पाँच मिनट मेइन् पहुन्च् रह हून्, तेन्सिओन् मत् लो.


In [12]:
# ---- listen to all three before deciding ------------------------------
# If section 5 said latin IS in the vocabulary, "raw latin" may well win and
# you should transliterate nothing. The model was trained on en+hi data in
# SOME consistent format; matching that format is what matters, not tidiness.
print(f"reminder from section 5: latin in vocabulary = {HAS_LATIN}\n")

_probe = TRICKY[0]
for label, txt in [
    ("raw latin ", _probe),
    ("v1 dict   ", normalise(_probe)),
    ("v2 translit", normalise_v2(_probe)),
]:
    ids = tokenizer.text_to_ids(txt)
    if not ids:
        print(f"  {label}: 0 tokens - nothing survives, skipping")
        continue
    w = synth(txt, length_scale=LENGTH_SCALE, do_normalise=False)
    p = f"{AUDIO_DIR}/norm_{label.strip().replace(' ', '_')}.wav"
    sf.write(p, w.astype(np.float32), sr)
    print(f"\n  {label}  {len(ids)} tokens, {len(w)/sr:.2f} s")
    print(f"    {txt}")
    display(Audio(p))

print("""
======================================================================
CHOOSE, then set USE_V2 below and re-run this cell.
  USE_V2 = True   -> transliterate romanised Hindi to Devanagari
  USE_V2 = False  -> keep the dictionary-only version
======================================================================
""")

USE_V2 = True          # <-- your decision goes here

if USE_V2 and HAVE_TRANSLIT:
    normalise = normalise_v2
    print("using v2 (dictionary + transliteration)")
else:
    print("using v1 (dictionary only)")

# everything downstream must see the SAME text the model will get at run time
CALIB_TEXTS = [normalise(t) for t in CALIB_TEXTS_RAW]
CALIB_LONG  = [normalise(t) for t in CALIB_LONG_RAW]
TEST_TEXT   = normalise(TEST_TEXT_RAW)
print(f"\nTEST_TEXT   -> {TEST_TEXT}")
print(f"CALIB[0]    -> {CALIB_TEXTS[0]}")

reminder from section 5: latin in vocabulary = True


  raw latin   56 tokens, 3.69 s
    Sector 21 mein 12 log fase hain, ambulance turant bhejo.



  v1 dict     62 tokens, 4.18 s
    सेक्टर इक्कीस mein बारह log fase hain, एम्बुलेंस turant bhejo.



  v2 translit  64 tokens, 3.73 s
    सेक्टर इक्कीस मेइन् बारह लोग् फ़से हैन्, एम्बुलेंस तुरन्त् भेजो.



CHOOSE, then set USE_V2 below and re-run this cell.
  USE_V2 = True   -> transliterate romanised Hindi to Devanagari
  USE_V2 = False  -> keep the dictionary-only version

using v2 (dictionary + transliteration)

TEST_TEXT   -> गोदाम नंबर तीन पे फायर लग गई है
CALIB[0]    -> मदद कीजिए, यहाँ आग लगी है


## 8. Export to ONNX

Two graphs, not one. FastPitch's duration predictor decides the output length at
run time, so the pair cannot be traced as a single fixed graph without freezing
the sentence length. Two sessions is also how sherpa-onnx runs two-stage TTS on
Android.

**The attention swap.** PyTorch's fused `nn.MultiheadAttention` reshapes using a
Python int taken from the traced input, so the tracer writes that length into the
graph as a literal and `dynamic_axes` cannot rescue it. Writing the same
attention longhand with `-1` on the time axis keeps it symbolic. Weights are
reused unchanged, and the cell proves numerical equivalence before exporting.

**`length_scale` as an input.** Bake in the ability to change speed, not the
speed itself. One file then serves both urgent alerts and normal messages.

In [13]:
import math, torch.nn as nn, torch.nn.functional as F
from collections import Counter

# reference output before patching, to prove the patch changes nothing
_x = torch.LongTensor(tokenizer.text_to_ids(TEST_TEXT)).unsqueeze(0)
_spk = torch.LongTensor([SPEAKER_ID])
with torch.no_grad():
    REF = tts_model.inference(_x, aux_input={"d_vectors": None,
                                             "speaker_ids": _spk})["model_outputs"].clone()

class ExplicitMHA(nn.Module):
    """Drop-in for nn.MultiheadAttention (batch_first=False, self-attention).
    Same weights, same maths, every reshape uses -1 on the time axis so it
    stays dynamic through the tracer."""
    def __init__(self, m):
        super().__init__()
        self.nh, self.e = m.num_heads, m.embed_dim
        self.dk = self.e // self.nh
        self.in_w = nn.Parameter(m.in_proj_weight.detach().clone())
        self.in_b = nn.Parameter(m.in_proj_bias.detach().clone())
        self.out = nn.Linear(self.e, self.e)
        self.out.weight = nn.Parameter(m.out_proj.weight.detach().clone())
        self.out.bias = nn.Parameter(m.out_proj.bias.detach().clone())

    def forward(self, q, k, v, attn_mask=None, key_padding_mask=None,
                need_weights=True, **kw):
        qkv = F.linear(q, self.in_w, self.in_b)
        qq, kk, vv = qkv.chunk(3, dim=-1)
        B, nh, dk = 1, self.nh, self.dk
        qq = qq.reshape(-1, B * nh, dk).transpose(0, 1)
        kk = kk.reshape(-1, B * nh, dk).transpose(0, 1)
        vv = vv.reshape(-1, B * nh, dk).transpose(0, 1)
        att = torch.softmax(qq @ kk.transpose(-2, -1) / math.sqrt(dk), dim=-1)
        o = (att @ vv).transpose(0, 1).contiguous().reshape(-1, B, self.e)
        return self.out(o), att.mean(0, keepdim=True)

def swap(mod):
    n = 0
    for name, child in list(mod.named_children()):
        if isinstance(child, nn.MultiheadAttention):
            setattr(mod, name, ExplicitMHA(child).eval())
            n += 1
        else:
            n += swap(child)
    return n

n = swap(tts_model)
tts_model.eval()
print(f"replaced {n} attention modules")

with torch.no_grad():
    now = tts_model.inference(_x, aux_input={"d_vectors": None,
                                             "speaker_ids": _spk})["model_outputs"]
d = float((REF - now).abs().max())
print(f"before vs after swap: max diff {d:.2e}  "
      f"{'OK' if d < 1e-3 else '*** MISMATCH - stop here ***'}")

replaced 12 attention modules
before vs after swap: max diff 3.19e-05  OK


In [14]:
class FastPitchONNX(nn.Module):
    """tokens + speaker_id + length_scale -> mel, shaped for HiFi-GAN.

    Body follows ForwardTTS.inference() with two changes:
      * the mask comes from ones_like(tokens), which derives from the INPUT and
        so stays dynamic, instead of from a Python int the tracer would freeze
      * durations are scaled by a tensor input rather than a config constant
    """
    def __init__(self, m, transpose):
        super().__init__()
        self.m, self.transpose = m, transpose

    def forward(self, tokens, speaker_id, length_scale):
        m = self.m
        x_mask = torch.ones_like(tokens).unsqueeze(1).float()
        o_en, x_mask, g, _ = m._forward_encoder(tokens, x_mask, speaker_id)
        o_dr_log = m.duration_predictor(o_en.squeeze(), x_mask)
        o_dr = m.format_durations(o_dr_log, x_mask).squeeze(1)
        o_dr = torch.clamp(torch.round(o_dr * length_scale), min=1.0)
        y_lengths = o_dr.sum(1)
        if m.args.use_pitch:
            e, _ = m._forward_pitch_predictor(o_en, x_mask); o_en = o_en + e
        if m.args.use_energy:
            e, _ = m._forward_energy_predictor(o_en, x_mask); o_en = o_en + e
        o_de, _ = m._forward_decoder(o_en, o_dr, x_mask, y_lengths, g=None)
        return o_de.transpose(1, 2) if self.transpose else o_de

class HiFiGANONNX(nn.Module):
    def __init__(self, g):
        super().__init__()
        self.g = g
    def forward(self, mel):
        return self.g(mel)

ls_t = torch.tensor([1.0], dtype=torch.float32)
fp_wrap = FastPitchONNX(tts_model, NEEDS_TRANSPOSE).eval()

with torch.no_grad():
    ref = tts_model.inference(_x, aux_input={"d_vectors": None,
                                             "speaker_ids": _spk})["model_outputs"]
    ref = ref.transpose(1, 2) if NEEDS_TRANSPOSE else ref
    got = fp_wrap(_x, _spk, ls_t)
if ref.shape == got.shape:
    dd = float((ref - got).abs().max())
    print(f"wrapper vs inference() @ ls=1.0: max diff {dd:.2e}  "
          f"{'OK' if dd < 1e-3 else 'CHECK'}")
else:
    print(f"frame count differs by {abs(ref.shape[-1]-got.shape[-1])} "
          f"(rounding). Small is fine.")

mel_v = got
hg_wrap = HiFiGANONNX(gen).eval()

print("\nexporting ...")
with torch.no_grad():
    torch.onnx.export(
        fp_wrap, (_x, _spk, ls_t), FP_ONNX,
        input_names=["tokens", "speaker_id", "length_scale"],
        output_names=["mel"],
        dynamic_axes={"tokens": {0: "B", 1: "T_text"},
                      "speaker_id": {0: "B"}, "mel": {0: "B", 2: "T_mel"}},
        opset_version=17, do_constant_folding=True,
        dynamo=False)          # torch>=2.6 defaults True and needs onnxscript
    print(f"  fastpitch -> {os.path.getsize(FP_ONNX)/1e6:.1f} MB")

    torch.onnx.export(
        hg_wrap, (mel_v,), HG_ONNX,
        input_names=["mel"], output_names=["wav"],
        dynamic_axes={"mel": {0: "B", 2: "T_mel"}, "wav": {0: "B", 2: "T_wav"}},
        opset_version=17, do_constant_folding=True,
        dynamo=False)          # the traced path is what ExplicitMHA targets
    print(f"  hifigan   -> {os.path.getsize(HG_ONNX)/1e6:.1f} MB")

def ops(p):
    return Counter(node.op_type for node in onnx.load(p).graph.node)

for label, p in [("fastpitch", FP_ONNX), ("hifigan", HG_ONNX)]:
    o = ops(p)
    print(f"\n{label}: {sum(o.values())} nodes  Conv={o.get('Conv',0)} "
          f"ConvTranspose={o.get('ConvTranspose',0)} "
          f"MatMul={o.get('MatMul',0)+o.get('Gemm',0)}")

wrapper vs inference() @ ls=1.0: max diff 0.00e+00  OK

exporting ...


/tmp/ipykernel_394/1493146184.py:56: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/TTS/tts/layers/generic/pos_encoding.py:47: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.pe.size(2) < x.size(2):
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:305: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >=

  fastpitch -> 217.2 MB


/tmp/ipykernel_394/1493146184.py:66: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


  hifigan   -> 55.7 MB

fastpitch: 792 nodes  Conv=32 ConvTranspose=0 MatMul=48

hifigan: 200 nodes  Conv=74 ConvTranspose=4 MatMul=0


In [15]:
# ---- the test the export must pass: variable length AND variable speed --
s_fp = ort.InferenceSession(FP_ONNX, providers=["CPUExecutionProvider"])
s_hg = ort.InferenceSession(HG_ONNX, providers=["CPUExecutionProvider"])

def onnx_synth(text, length_scale=1.0, fp=None, hg=None):
    a, b = fp or s_fp, hg or s_hg
    ids = tokenizer.text_to_ids(normalise(text))
    mel = a.run(None, {"tokens": np.array([ids], np.int64),
                       "speaker_id": np.array([SPEAKER_ID], np.int64),
                       "length_scale": np.array([length_scale], np.float32)})[0]
    return np.asarray(b.run(None, {"mel": mel})[0]).squeeze(), mel.shape[-1]

print("variable LENGTH (different sentences must give different durations):")
for t in ["मदद कीजिए।",
          "गोदाम नंबर तीन पे फायर लग गई है।",
          "सेक्टर इक्कीस में बारह लोग फँसे हैं, एम्बुलेंस तुरंत भेजिए।"]:
    w, frames = onnx_synth(t)
    print(f"   {len(tokenizer.text_to_ids(normalise(t))):3d} tokens -> "
          f"{frames:4d} frames -> {len(w)/sr:.2f} s   {t[:34]}")

print("\nvariable SPEED (same sentence, different length_scale):")
for v in [0.8, 0.9, 1.0, 1.2]:
    w, frames = onnx_synth(TEST_TEXT, length_scale=v)
    print(f"   ls={v:.1f}  {frames:4d} frames  {len(w)/sr:.2f} s")

print("""
Both lists must vary. If every row is identical, a shape got frozen into the
graph and the attention swap did not take -- re-run the previous two cells.
""")

variable LENGTH (different sentences must give different durations):
     9 tokens ->   61 frames -> 0.71 s   मदद कीजिए।
    31 tokens ->  206 frames -> 2.39 s   गोदाम नंबर तीन पे फायर लग गई है।
    58 tokens ->  319 frames -> 3.70 s   सेक्टर इक्कीस में बारह लोग फँसे है

variable SPEED (same sentence, different length_scale):
   ls=0.8   166 frames  1.93 s
   ls=0.9   183 frames  2.12 s
   ls=1.0   206 frames  2.39 s
   ls=1.2   246 frames  2.86 s

Both lists must vary. If every row is identical, a shape got frozen into the
graph and the attention swap did not take -- re-run the previous two cells.



## 9. Quantise — static QDQ, not dynamic

On a conv-heavy model, plain `quantize_dynamic` emits one `ConvInteger` node per
convolution. ONNX Runtime ships **no ARM kernel** for that operator, so the file
gets smaller, the script exits zero, and the model then fails to load on the
phone.

Static QDQ emits `QuantizeLinear`/`DequantizeLinear` pairs that fuse into
`QLinearConv`, which does have ARM kernels. It needs calibration data, which is
why people avoid it — and why section 1 defined eight real sentences.

The `ConvInteger == 0` assertion at the end is a hard deployment gate.

In [16]:
from onnxruntime.quantization import (quantize_static, QuantType, QuantFormat,
                                      CalibrationDataReader)

class TokenCalib(CalibrationDataReader):
    def __init__(self, texts):
        self.data = iter([
            {"tokens": np.array([tokenizer.text_to_ids(t)], np.int64),
             "speaker_id": np.array([SPEAKER_ID], np.int64),
             "length_scale": np.array([LENGTH_SCALE], np.float32)}
            for t in texts])
    def get_next(self):
        return next(self.data, None)

class MelCalib(CalibrationDataReader):
    """Calibrate the vocoder on mels FastPitch really produces, never on noise.
    Activation ranges must come from the real distribution."""
    def __init__(self, texts):
        s = ort.InferenceSession(FP_ONNX, providers=["CPUExecutionProvider"])
        mels = []
        for t in texts:
            mels.append({"mel": s.run(None, {
                "tokens": np.array([tokenizer.text_to_ids(t)], np.int64),
                "speaker_id": np.array([SPEAKER_ID], np.int64),
                "length_scale": np.array([LENGTH_SCALE], np.float32)})[0]})
        self.data = iter(mels)
    def get_next(self):
        return next(self.data, None)

for src, dst, reader, label in [
    (FP_ONNX, FP_Q, TokenCalib(CALIB_TEXTS), "fastpitch"),
    (HG_ONNX, HG_Q, MelCalib(CALIB_TEXTS),   "hifigan"),
]:
    print(f"quantising {label} on {len(CALIB_TEXTS)} sentences ...")
    try:
        quantize_static(src, dst, reader,
                        quant_format=QuantFormat.QDQ,
                        activation_type=QuantType.QUInt8,
                        weight_type=QuantType.QInt8,
                        per_channel=True)
        a = os.path.getsize(src) / 1e6
        b = os.path.getsize(dst) / 1e6
        print(f"   {a:.1f} MB -> {b:.1f} MB  ({a/b:.2f}x smaller)")
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {e}")

print("\n" + "=" * 64)
print("THE GATE - ConvInteger must be 0 or it will not load on Android")
print("=" * 64)
bad = False
for label, p in [("fastpitch int8", FP_Q), ("hifigan int8", HG_Q)]:
    if not os.path.exists(p):
        print(f"  {label}: not produced")
        continue
    o = ops(p)
    ci = o.get("ConvInteger", 0)
    bad |= ci > 0
    print(f"  {label:16s} ConvInteger={ci:<4d} QLinearConv={o.get('QLinearConv',0):<4d} "
          f"QuantizeLinear={o.get('QuantizeLinear',0):<5d} "
          f"MatMulInteger={o.get('MatMulInteger',0)}")
print("\n  " + ("FAIL - do not ship this" if bad else "PASS - zero ConvInteger"))

quantising fastpitch on 28 sentences ...


   217.2 MB -> 63.1 MB  (3.44x smaller)
quantising hifigan on 28 sentences ...


   55.7 MB -> 14.3 MB  (3.89x smaller)

THE GATE - ConvInteger must be 0 or it will not load on Android
  fastpitch int8   ConvInteger=0    QLinearConv=0    QuantizeLinear=435   MatMulInteger=0
  hifigan int8     ConvInteger=0    QLinearConv=0    QuantizeLinear=196   MatMulInteger=0

  PASS - zero ConvInteger


## 9b. Was that int8 damage real?

The naive comparison writes `a32[:n]` against `a8[:n]` and calls the difference
quantisation damage. It is not.

If int8 FastPitch predicts even slightly different durations, the two waveforms
drift out of alignment after the first token that changed. From that point you
are comparing "sec-" against "-tor" and the number explodes even when both
sound perfect. A 4.18 s vs 4.31 s pair is already 3% out of phase.

So ask two separate questions:

**Q1 — did quantising FastPitch move the durations?** Compare mel frame counts.
Pure length, no waveform involved.

**Q2 — did quantising HiFi-GAN damage the waveform?** Feed **the same fp32 mel**
to both vocoders. Identical input, identical length, so any difference is
genuinely the vocoder.

Q2 is the number that means something.

In [17]:
s_fp32 = ort.InferenceSession(FP_ONNX, providers=["CPUExecutionProvider"])
s_hg32 = ort.InferenceSession(HG_ONNX, providers=["CPUExecutionProvider"])
s_fp8  = ort.InferenceSession(FP_Q,    providers=["CPUExecutionProvider"])
s_hg8  = ort.InferenceSession(HG_Q,    providers=["CPUExecutionProvider"])

def mel_of(sess, text, ls=None):
    ids = tokenizer.text_to_ids(normalise(text))
    return sess.run(None, {
        "tokens": np.array([ids], np.int64),
        "speaker_id": np.array([SPEAKER_ID], np.int64),
        "length_scale": np.array([ls or LENGTH_SCALE], np.float32)})[0]

def wav_of(sess, mel):
    return np.asarray(sess.run(None, {"mel": mel})[0]).squeeze()

def logspec_l1(a, b):
    n = min(len(a), len(b))
    A = np.log(np.abs(librosa.stft(a[:n].astype(np.float32))) + 1e-8)
    B = np.log(np.abs(librosa.stft(b[:n].astype(np.float32))) + 1e-8)
    return float(np.abs(A - B).mean())

PROBE = [
    "मदद कीजिए।",
    "गोदाम नंबर 3 पे फायर लग गई है।",
    "Sector 21 mein 12 log fase hain, ambulance turant bhejo.",
]

print("=" * 70)
print("Q1  FASTPITCH - did int8 change the predicted durations?")
print("=" * 70)
print(f"{'text':36s} {'fp32':>8s} {'int8':>8s} {'drift':>8s}")
drifts = []
for t in PROBE:
    f32 = mel_of(s_fp32, t).shape[-1]
    f8  = mel_of(s_fp8,  t).shape[-1]
    d = 100 * (f8 - f32) / f32
    drifts.append(abs(d))
    print(f"{t[:36]:36s} {f32:6d}fr {f8:6d}fr {d:+7.1f}%")
MEAN_DRIFT = float(np.mean(drifts))
print(f"\n  mean |drift| = {MEAN_DRIFT:.1f}%")
print("  under ~1% is rounding noise; over ~2% the duration predictor is")
print("  being damaged and section 9c fixes it.")

print("\n" + "=" * 70)
print("Q2  HIFI-GAN - waveform damage, measured on IDENTICAL mels")
print("=" * 70)
l1s = []
for t in PROBE:
    m32 = mel_of(s_fp32, t)                    # same mel to both vocoders
    l1 = logspec_l1(wav_of(s_hg32, m32), wav_of(s_hg8, m32))
    l1s.append(l1)
    verdict = ("inaudible" if l1 < 0.5 else
               "audible, probably fine" if l1 < 1.5 else "DAMAGED")
    print(f"  {t[:42]:42s} L1 {l1:.3f}   {verdict}")
MEAN_L1 = float(np.mean(l1s))

print(f"""
  mean vocoder L1 = {MEAN_L1:.3f}

  This is the honest number. If it is under 0.5 the vocoder is fine and any
  large figure you saw earlier was duration drift, not damage.
""")

Q1  FASTPITCH - did int8 change the predicted durations?
text                                     fp32     int8    drift
मदद कीजिए।                               54fr     49fr    -9.3%
गोदाम नंबर 3 पे फायर लग गई है।          183fr    192fr    +4.9%
Sector 21 mein 12 log fase hain, amb    318fr    332fr    +4.4%

  mean |drift| = 6.2%
  under ~1% is rounding noise; over ~2% the duration predictor is
  being damaged and section 9c fixes it.

Q2  HIFI-GAN - waveform damage, measured on IDENTICAL mels
  मदद कीजिए।                                 L1 1.334   audible, probably fine
  गोदाम नंबर 3 पे फायर लग गई है।             L1 1.676   DAMAGED
  Sector 21 mein 12 log fase hain, ambulance L1 1.436   audible, probably fine

  mean vocoder L1 = 1.482

  This is the honest number. If it is under 0.5 the vocoder is fine and any
  large figure you saw earlier was duration drift, not damage.



## 9c. Re-quantise, protecting what int8 cannot take

Two parts of this pipeline tolerate int8 badly, and both are cheap to exclude
because neither holds much of the weight:

**The duration, pitch and energy predictors.** Tiny conv stacks whose output goes
through `round()`. A numeric error of 0.4 frames flips a token from 7 frames to
8, and those errors accumulate across a sentence. That is the drift Q1 measured.

**HiFi-GAN's `ConvTranspose` layers.** The upsampling path carries the waveform's
full dynamic range and is the classic weak point for int8 in vocoders.

Excluding both costs a few megabytes and usually removes most of the damage.

In [18]:
def nodes_matching(path, substrings=(), op_types=()):
    m = onnx.load(path)
    out = []
    for n in m.graph.node:
        low = n.name.lower()
        if any(s in low for s in substrings) or n.op_type in op_types:
            out.append(n.name)
    return sorted(set(out))

FP_EXCL = nodes_matching(FP_ONNX, substrings=("duration", "pitch", "energy"))
HG_EXCL = nodes_matching(HG_ONNX, substrings=("convtranspose", "ups"),
                         op_types=("ConvTranspose",))

print(f"excluding from fastpitch : {len(FP_EXCL)} nodes")
print(f"excluding from hifigan   : {len(HG_EXCL)} nodes")

FP_Q2 = FP_ONNX.replace(".onnx", ".qdq-safe.onnx")
HG_Q2 = HG_ONNX.replace(".onnx", ".qdq-safe.onnx")

ok = True
for src, dst, reader, excl, label in [
    (FP_ONNX, FP_Q2, TokenCalib(CALIB_TEXTS), FP_EXCL, "fastpitch"),
    (HG_ONNX, HG_Q2, MelCalib(CALIB_TEXTS),   HG_EXCL, "hifigan"),
]:
    print(f"\nquantising {label} on {len(CALIB_TEXTS)} sentences ...")
    try:
        quantize_static(src, dst, reader,
                        quant_format=QuantFormat.QDQ,
                        activation_type=QuantType.QUInt8,
                        weight_type=QuantType.QInt8,
                        per_channel=True,
                        nodes_to_exclude=excl)
        a, b = os.path.getsize(src)/1e6, os.path.getsize(dst)/1e6
        print(f"   {a:.1f} MB -> {b:.1f} MB  ({a/b:.2f}x smaller)")
    except Exception as e:
        ok = False
        print(f"   FAILED: {type(e).__name__}: {e}")

# ---- the gate, again ---------------------------------------------------
print("\n" + "=" * 70)
print("THE GATE - ConvInteger must be 0 or it will not load on Android")
print("=" * 70)
bad = False
for label, p in [("fastpitch safe", FP_Q2), ("hifigan safe", HG_Q2)]:
    if not os.path.exists(p):
        print(f"  {label}: not produced")
        ok = False
        continue
    o = ops(p)
    ci = o.get("ConvInteger", 0)
    bad |= ci > 0
    print(f"  {label:16s} ConvInteger={ci:<4d} QLinearConv={o.get('QLinearConv',0):<4d} "
          f"QuantizeLinear={o.get('QuantizeLinear',0):<5d}")
print("\n  " + ("FAIL - do not ship" if bad else "PASS - zero ConvInteger"))

# ---- did it help? ------------------------------------------------------
if ok and not bad:
    s_fp8b = ort.InferenceSession(FP_Q2, providers=["CPUExecutionProvider"])
    s_hg8b = ort.InferenceSession(HG_Q2, providers=["CPUExecutionProvider"])
    print("\n" + "=" * 70)
    print("BEFORE vs AFTER the exclusions")
    print("=" * 70)
    print(f"{'text':36s} {'drift':>9s} {'vocoder L1':>12s}")
    nd, nl = [], []
    for t in PROBE:
        m32 = mel_of(s_fp32, t)
        d = 100 * (mel_of(s_fp8b, t).shape[-1] - m32.shape[-1]) / m32.shape[-1]
        l1 = logspec_l1(wav_of(s_hg32, m32), wav_of(s_hg8b, m32))
        nd.append(abs(d)); nl.append(l1)
        print(f"{t[:36]:36s} {d:+8.1f}% {l1:12.3f}")
    print(f"\n  drift      {MEAN_DRIFT:.1f}%  ->  {np.mean(nd):.1f}%")
    print(f"  vocoder L1 {MEAN_L1:.3f}  ->  {np.mean(nl):.3f}")

    # adopt the safer pair everywhere downstream
    FP_Q, HG_Q = FP_Q2, HG_Q2
    print(f"\n  adopted: FP_Q -> {os.path.basename(FP_Q)}")
    print(f"           HG_Q -> {os.path.basename(HG_Q)}")
    print("""
  If vocoder L1 is still above ~1.5, the next lever is calibration data:
  go from 8 sentences to 30-40 covering more of the phonetic space, then
  re-run this cell.
""")
else:
    print("\n  keeping the earlier int8 pair; fix the failure above first.")

excluding from fastpitch : 71 nodes
excluding from hifigan   : 4 nodes

quantising fastpitch on 28 sentences ...


   217.2 MB -> 66.6 MB  (3.26x smaller)

quantising hifigan on 28 sentences ...


   55.7 MB -> 22.3 MB  (2.50x smaller)

THE GATE - ConvInteger must be 0 or it will not load on Android
  fastpitch safe   ConvInteger=0    QLinearConv=0    QuantizeLinear=389  
  hifigan safe     ConvInteger=0    QLinearConv=0    QuantizeLinear=196  

  PASS - zero ConvInteger

BEFORE vs AFTER the exclusions
text                                     drift   vocoder L1
मदद कीजिए।                               +1.9%        1.334
गोदाम नंबर 3 पे फायर लग गई है।           +3.3%        1.678
Sector 21 mein 12 log fase hain, amb     +2.8%        1.444

  drift      6.2%  ->  2.7%
  vocoder L1 1.482  ->  1.485

  adopted: FP_Q -> fastpitch_en+hi.qdq-safe.onnx
           HG_Q -> hifigan_en+hi.qdq-safe.onnx

  If vocoder L1 is still above ~1.5, the next lever is calibration data:
  go from 8 sentences to 30-40 covering more of the phonetic space, then
  re-run this cell.



## 9d. Calibration is why L1 is high

An L1 of 1.6 means audible damage. Three causes, in order of how much they
usually matter:

**1. MinMax calibration is outlier-driven.** The default method takes the
smallest and largest activation it saw and maps the whole int8 range across it.
One loud plosive sets the maximum, and every quiet consonant afterwards gets
squeezed into a handful of levels. `Percentile` clips the top 0.001% instead, so
the range covers what the signal actually spends its time doing. On audio this
is usually worth more than every other adjustment combined.

**2. Too few calibration sentences.** Eight cannot cover the phonetic space of a
language. Thirty-plus, including long ones, gives the calibrator a real
distribution to measure.

**3. No long sentences in the calibration set.** This is also why long audio
breaks — see section 10a. Activations grow with sequence length, so if the
calibrator only saw four-second clips it has no idea how large things get at
twelve seconds, and everything past its learned ceiling saturates.

This cell tries four methods and measures each. Pick the winner by number.

In [19]:
# ── 9d: calibration sweep, memory-bounded ──────────────────────────────
# Histogram calibrators (Percentile, Entropy) dump EVERY intermediate tensor
# for EVERY sample, then histogram them. On a vocoder those intermediates are
# waveform-scale, so the memory cost is samples x nodes x seconds. Three
# guards below: a small subset, short sentences only, one method at a time.
import gc, psutil
from onnxruntime.quantization import CalibrationMethod

def ram():
    m = psutil.virtual_memory()
    return f"{m.available/1e9:.1f} GB free of {m.total/1e9:.1f}"

print(f"RAM: {ram()}\n")

# 1. SHORT sentences only, and few of them. Histogram cost scales with audio
#    duration, and the long ones are what blew the kernel up.
def est_frames(t):
    return len(tokenizer.text_to_ids(normalise(t)))

CALIB_HIST = sorted([t for t in CALIB_TEXTS if est_frames(t) <= 40],
                    key=est_frames)[:8]
print(f"histogram calibration subset: {len(CALIB_HIST)} short sentences")
for t in CALIB_HIST:
    print(f"   {est_frames(t):3d} tok  {t[:44]}")

# 2. MinMax keeps the full set - it is a running min/max, near-zero memory.
print(f"\nminmax calibration set: {len(CALIB_TEXTS)} sentences "
      f"(incl. {len(CALIB_LONG)} long)")


def quant_one(name, method, pct, texts):
    """One method, one process-lifetime, cleaned up after."""
    dst = HG_ONNX.replace(".onnx", f".{name}.onnx")
    kw = dict(quant_format=QuantFormat.QDQ,
              activation_type=QuantType.QUInt8,
              weight_type=QuantType.QInt8,
              per_channel=True,
              nodes_to_exclude=HG_EXCL,
              calibrate_method=method)
    extra = {}
    if pct is not None:
        extra["CalibPercentile"] = pct
    # caps how many intermediate tensors are held at once; ignored by older
    # onnxruntime, which is why it goes in a try
    extra["CalibMaxIntermediateOutputs"] = 24
    if extra:
        kw["extra_options"] = extra

    for attempt in (kw, {k: v for k, v in kw.items() if k != "extra_options"}):
        try:
            quantize_static(HG_ONNX, dst, MelCalib(texts), **attempt)
            gc.collect()
            return dst
        except MemoryError:
            print(f"   {name}: MemoryError - skipping")
            return None
        except Exception as e:
            if attempt is kw:
                continue          # retry without extra_options
            print(f"   {name}: {type(e).__name__}: {str(e)[:90]}")
            return None
    return None


def score(path):
    s = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    l1s = [logspec_l1(wav_of(s_hg32, mel_of(s_fp32, t)),
                      wav_of(s,      mel_of(s_fp32, t)))
           for t in PROBE + CALIB_LONG[:1]]
    del s
    gc.collect()
    return float(np.mean(l1s))


# 3. One at a time, with cleanup between. Entropy is the heaviest and the
#    least useful for audio - leave it off unless you have RAM to burn.
PLAN = [
    ("minmax",          CalibrationMethod.MinMax,     None, CALIB_TEXTS),
    ("percentile-999",  CalibrationMethod.Percentile, 99.9,  CALIB_HIST),
    ("percentile-9999", CalibrationMethod.Percentile, 99.99, CALIB_HIST),
    # ("entropy",       CalibrationMethod.Entropy,    None,  CALIB_HIST[:4]),
]

results = {}
for name, method, pct, texts in PLAN:
    print(f"\n{name}  ({len(texts)} samples)  RAM {ram()}")
    dst = quant_one(name, method, pct, texts)
    gc.collect()
    if not dst or not os.path.exists(dst):
        continue
    l1 = score(dst)
    mb = os.path.getsize(dst) / 1e6
    results[name] = (l1, dst, mb)
    verdict = ("inaudible" if l1 < 0.5 else
               "probably fine" if l1 < 1.5 else "DAMAGED")
    print(f"   L1 {l1:.3f}   {mb:.1f} MB   {verdict}")

print("\n" + "=" * 66)
if results:
    best = min(results, key=lambda k: results[k][0])
    l1, path, mb = results[best]
    for k, (v, _, m) in sorted(results.items(), key=lambda kv: kv[1][0]):
        print(f"  {'->' if k == best else '  '} {k:18s} L1 {v:.3f}   {m:6.1f} MB")
    HG_Q = path
    print(f"\n  HG_Q -> {os.path.basename(HG_Q)}")
    if l1 > 1.5:
        print("""
  Still damaged. In order of what actually helps:
    1. widen HG_EXCL - add the first and last Conv, not just ConvTranspose
    2. ship HiFi-GAN in fp32. It is ~55 MB and it is the half that decides
       how the output sounds. FastPitch int8 alone still saves ~140 MB.
       That is a completely reasonable place to stop.
""")
else:
    print("  every method failed; keeping the existing HG_Q")
print("=" * 66)

RAM: 29.5 GB free of 33.7

histogram calibration subset: 8 short sentences
    12 tok  जल्दी कीजिए!
    15 tok  बिजली काट दीजिए
    15 tok  कौन बोल रहा है?
    17 tok  पानी और दवा भेजिए
    17 tok  यहाँ बहुत धुआँ है
    20 tok  रस्सी और टॉर्च चाहिए
    21 tok  दो लोग घायल हो गए हैं
    21 tok  गोदाम नंबर तीन पर आइए

minmax calibration set: 28 sentences (incl. 4 long)

minmax  (28 samples)  RAM 29.5 GB free of 33.7


   L1 1.523   22.3 MB   DAMAGED

percentile-999  (8 samples)  RAM 29.4 GB free of 33.7


   percentile-999: ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape a

percentile-9999  (8 samples)  RAM 28.9 GB free of 33.7


   percentile-9999: ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape a

  -> minmax             L1 1.523     22.3 MB

  HG_Q -> hifigan_en+hi.minmax.onnx

  Still damaged. In order of what actually helps:
    1. widen HG_EXCL - add the first and last Conv, not just ConvTranspose
    2. ship HiFi-GAN in fp32. It is ~55 MB and it is the half that decides
       how the output sounds. FastPitch int8 alone still saves ~140 MB.
       That is a completely reasonable place to stop.



## 10a. Why long sentences break

Something fails past roughly four seconds. Three candidates, and they need
different fixes, so measure before changing anything:

| If it breaks in | Cause | Fix |
|---|---|---|
| PyTorch too | model limit | chunking (10b) |
| fp32 ONNX only | a shape froze during export | re-export with a long dummy input |
| int8 ONNX only | calibration ceiling | 9d, with long sentences |

This runs the same growing text through all three and reports the first length
where each diverges from the PyTorch reference.

In [20]:
def build_text(n_clauses):
    base = ["मदद कीजिए", "गोदाम नंबर तीन पे आग लगी है",
            "दो लोग घायल हैं", "पानी और दवा भेजिए",
            "रास्ता बंद है", "पाँच मिनट में पहुँच रहा हूँ",
            "सब सुरक्षित हैं", "एम्बुलेंस तुरंत भेजिए"]
    return ", ".join(base[i % len(base)] for i in range(n_clauses)) + "।"

def pt_wav(text):
    return synth(text, length_scale=LENGTH_SCALE, do_normalise=True)

def onnx_wav(text, fp, hg):
    ids = tokenizer.text_to_ids(normalise(text))
    mel = fp.run(None, {"tokens": np.array([ids], np.int64),
                        "speaker_id": np.array([SPEAKER_ID], np.int64),
                        "length_scale": np.array([LENGTH_SCALE], np.float32)})[0]
    return np.asarray(hg.run(None, {"mel": mel})[0]).squeeze()

def health(w, sr):
    """Cheap signal-quality checks. Broken synthesis is usually silent,
    saturated, or non-finite - all three show up here."""
    if w is None or len(w) == 0:
        return "EMPTY"
    if not np.all(np.isfinite(w)):
        return "NaN/Inf"
    peak, rms = float(np.abs(w).max()), float(np.sqrt((w ** 2).mean()))
    if peak < 1e-3:
        return "SILENT"
    if peak > 0.999 and (np.abs(w) > 0.99).mean() > 0.05:
        return "CLIPPED"
    if rms < 1e-4:
        return "near-silent"
    return "ok"

s_int8_fp = ort.InferenceSession(FP_Q, providers=["CPUExecutionProvider"]) \
            if os.path.exists(FP_Q) else None
s_int8_hg = ort.InferenceSession(HG_Q, providers=["CPUExecutionProvider"]) \
            if os.path.exists(HG_Q) else None

print(f"{'clauses':>8s} {'tokens':>7s} {'PyTorch':>22s} {'fp32 ONNX':>22s} "
      f"{'int8 ONNX':>22s}")
print("-" * 88)

FIRST_BAD = {}
for n in [1, 2, 4, 6, 8, 12, 16]:
    text = build_text(n)
    ids = tokenizer.text_to_ids(normalise(text))
    row = [f"{n:8d} {len(ids):7d}"]
    for label, fn in [
        ("pt",   lambda: pt_wav(text)),
        ("fp32", lambda: onnx_wav(text, s_fp32, s_hg32)),
        ("int8", (lambda: onnx_wav(text, s_int8_fp, s_int8_hg))
                 if s_int8_fp and s_int8_hg else None),
    ]:
        if fn is None:
            row.append(f"{'-':>22s}"); continue
        try:
            w = fn()
            st = health(w, sr)
            row.append(f"{st + ' ' + format(len(w)/sr, '.1f') + 's':>22s}")
            if st != "ok" and label not in FIRST_BAD:
                FIRST_BAD[label] = (n, len(ids), st)
        except Exception as e:
            msg = f"{type(e).__name__}"[:20]
            row.append(f"{msg:>22s}")
            if label not in FIRST_BAD:
                FIRST_BAD[label] = (n, len(ids), msg)
    print(" ".join(row))

print("\n" + "=" * 72)
if not FIRST_BAD:
    print("  Nothing broke up to 16 clauses. The limit is elsewhere -")
    print("  check whether AudioTrack on the phone is the real bottleneck.")
else:
    for k, (n, ntok, why) in FIRST_BAD.items():
        print(f"  {k:5s} first failed at {n} clauses / {ntok} tokens  ({why})")
    ks = set(FIRST_BAD)
    if "pt" in ks:
        print("\n  -> PyTorch fails too, so this is a MODEL limit, not export.")
        print("     Chunking (10b) is the fix, and you need it for the app anyway.")
    elif "fp32" in ks:
        print("\n  -> fp32 ONNX fails but PyTorch does not: a shape froze during")
        print("     export. Re-run section 8 with a LONG dummy input (see below).")
    elif "int8" in ks:
        print("\n  -> only int8 fails: the calibration ceiling. Section 9d with")
        print("     CALIB_LONG included is the fix.")
print("=" * 72)

 clauses  tokens                PyTorch              fp32 ONNX              int8 ONNX
----------------------------------------------------------------------------------------
       1       9                ok 0.6s                ok 0.6s                ok 0.6s
       2      38                ok 3.0s                ok 3.0s                ok 3.0s
       4      74                ok 5.7s                ok 5.7s                ok 5.7s
       6     118                ok 8.5s                ok 8.6s                ok 8.8s
       8     158               ok 10.8s               ok 10.7s               ok 11.0s
      12     234               ok 16.6s               ok 16.5s               ok 16.8s
      16     318               ok 21.7s               ok 21.7s               ok 22.3s

  Nothing broke up to 16 clauses. The limit is elsewhere -
  check whether AudioTrack on the phone is the real bottleneck.


## 10b. Chunked synthesis — the fix, and what the app needs anyway

Whatever 10a blamed, chunking solves it, and you were going to build this
regardless: synthesising a whole sentence before playing anything means the
listener waits for the full render. Chunked, they hear clause one while clause
two is still being generated, so perceived latency is the first chunk only.

Two details that matter more than they look:

**Do not crossfade.** Crossfading synthesised speech creates phase artifacts.
FastPitch produces clean boundaries at clause breaks — trim ~20 ms of silence
from each edge and butt-join them. Simpler and it sounds better.

**Splitting needs a fallback.** Punctuation first, but a clause can still exceed
the limit, so fall back to word boundaries. In the app there is no punctuation at
all — CTC output has none — which is why the sender should transmit VAD pause
positions instead.

In [21]:
MAX_CHUNK_CHARS = 70          # tune from what 10a found; lower is safer

def split_clauses(text, max_chars=MAX_CHUNK_CHARS):
    """Punctuation boundaries first, then word boundaries as a fallback."""
    parts = [p for p in re.split(r"(?<=[।.!?,;:])\s*", text) if p.strip()]
    out = []
    for p in parts:
        if len(p) <= max_chars:
            out.append(p.strip()); continue
        words, cur = p.split(), ""
        for w in words:
            if len(cur) + len(w) + 1 > max_chars and cur:
                out.append(cur.strip()); cur = w
            else:
                cur = f"{cur} {w}".strip()
        if cur:
            out.append(cur.strip())
    return out


def synth_long(text, fp=None, hg=None, length_scale=None,
               gap_ms=60, trim_edges=True, verbose=True):
    """Synthesise clause by clause and butt-join. Returns (wav, timings).

    timings[i] is the seconds spent on chunk i. timings[0] is the number your
    PS actually scores: time-to-first-audio.
    """
    fp = fp or s_fp32
    hg = hg or s_hg32
    ls = length_scale or LENGTH_SCALE
    chunks = split_clauses(normalise(text))
    gap = np.zeros(int(gap_ms * sr / 1000), np.float32)

    pieces, timings = [], []
    for i, ch in enumerate(chunks):
        ids = tokenizer.text_to_ids(ch)
        if not ids:
            continue
        t0 = time.perf_counter()
        mel = fp.run(None, {"tokens": np.array([ids], np.int64),
                            "speaker_id": np.array([SPEAKER_ID], np.int64),
                            "length_scale": np.array([ls], np.float32)})[0]
        w = np.asarray(hg.run(None, {"mel": mel})[0]).squeeze()
        timings.append(time.perf_counter() - t0)
        if trim_edges:
            w, _ = librosa.effects.trim(w.astype(np.float32), top_db=40)
        if verbose:
            print(f"   [{i+1}/{len(chunks)}] {len(ids):3d} tok -> "
                  f"{len(w)/sr:5.2f}s in {timings[-1]*1000:5.0f}ms   {ch[:44]}")
        pieces.append(w)
        if i < len(chunks) - 1:
            pieces.append(gap)

    if not pieces:
        return np.zeros(0, np.float32), []
    return np.concatenate(pieces).astype(np.float32), timings


# ---- prove it on something that was failing --------------------------
LONG = build_text(10)
print(f"{len(LONG)} chars, {len(tokenizer.text_to_ids(normalise(LONG)))} tokens")
print(f"split into {len(split_clauses(normalise(LONG)))} chunks\n")

w, timings = synth_long(LONG)
total = len(w) / sr
print(f"\n  audio            {total:.2f} s")
print(f"  synthesis        {sum(timings):.2f} s   RTF {sum(timings)/total:.3f}")
print(f"  time to FIRST    {timings[0]*1000:.0f} ms   <- what the user feels")
print(f"  naive whole-sentence wait would be {sum(timings)*1000:.0f} ms")

p = f"{AUDIO_DIR}/long_chunked.wav"
sf.write(p, w, sr)
display(Audio(p))

print("""
If this plays cleanly while section 10a showed a failure, chunking has solved
it. Lower MAX_CHUNK_CHARS if you still hear artifacts at the joins; raise
gap_ms if the clauses run together.
""")

199 chars, 198 tokens
split into 10 chunks

   [1/10]  10 tok ->  0.85s in   386ms   मदद कीजिए,
   [2/10]  28 tok ->  2.12s in   917ms   गोदाम नंबर तीन पे आग लगी है,
   [3/10]  16 tok ->  1.07s in   464ms   दो लोग घायल हैं,
   [4/10]  18 tok ->  1.37s in   594ms   पानी और दवा भेजिए,
   [5/10]  14 tok ->  1.06s in   456ms   रास्ता बंद है,
   [6/10]  28 tok ->  1.57s in   666ms   पाँच मिनट में पहुँच रहा हूँ,
   [7/10]  16 tok ->  0.94s in   411ms   सब सुरक्षित हैं,
   [8/10]  22 tok ->  1.46s in   622ms   एम्बुलेंस तुरंत भेजिए,
   [9/10]  10 tok ->  0.85s in   383ms   मदद कीजिए,
   [10/10]  27 tok ->  2.08s in   886ms   गोदाम नंबर तीन पे आग लगी है

  audio            13.90 s
  synthesis        5.78 s   RTF 0.416
  time to FIRST    386 ms   <- what the user feels
  naive whole-sentence wait would be 5784 ms



If this plays cleanly while section 10a showed a failure, chunking has solved
it. Lower MAX_CHUNK_CHARS if you still hear artifacts at the joins; raise
gap_ms if the clauses run together.



## 10. Latency, measured honestly

Two things most benchmarks get wrong:

**Session construction is not per-utterance cost.** Building an
`InferenceSession` takes hundreds of milliseconds and happens once at app
startup. Counting it per sentence makes your model look far worse than it is.

**Kaggle has many fast cores; your target phone does not.** `intra_op_num_threads
= 2` is a rough mid-range proxy. Treat everything here as an upper bound and
re-measure on the device.

Target: **RTF under 0.3**, so chunked playback stays ahead of the listener.
Above 1.0 the buffer drains faster than you fill it and chunking cannot help.

In [22]:
def make_sessions(fp_path, hg_path, threads=2):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    t0 = time.perf_counter()
    a = ort.InferenceSession(fp_path, so, providers=["CPUExecutionProvider"])
    b = ort.InferenceSession(hg_path, so, providers=["CPUExecutionProvider"])
    return a, b, time.perf_counter() - t0

BENCH = [
    "मदद कीजिए।",
    "गोदाम नंबर 3 पे फायर लग गई है।",
    "Sector 21 mein 12 log fase hain, ambulance turant bhejo.",
]

def bench(fp_path, hg_path, repeats=5):
    a, b, load_s = make_sessions(fp_path, hg_path)
    sd = np.array([SPEAKER_ID], np.int64)
    lsv = np.array([LENGTH_SCALE], np.float32)
    tk = np.array([tokenizer.text_to_ids(normalise(BENCH[0]))], np.int64)
    b.run(None, {"mel": a.run(None, {"tokens": tk, "speaker_id": sd,
                                     "length_scale": lsv})[0]})   # warm up
    rows = []
    for text in BENCH:
        tk = np.array([tokenizer.text_to_ids(normalise(text))], np.int64)
        fps, hgs, wav = [], [], None
        for _ in range(repeats):
            t0 = time.perf_counter()
            mel = a.run(None, {"tokens": tk, "speaker_id": sd,
                               "length_scale": lsv})[0]
            t1 = time.perf_counter()
            wav = b.run(None, {"mel": mel})[0]
            t2 = time.perf_counter()
            fps.append(t1 - t0); hgs.append(t2 - t1)
        w = np.asarray(wav).squeeze()
        dur = len(w) / sr
        f_ms, h_ms = np.median(fps) * 1000, np.median(hgs) * 1000
        rows.append((text[:30], f_ms, h_ms, dur, (f_ms + h_ms) / 1000 / dur))
    return load_s, rows

for label, fp_p, hg_p in [("fp32", FP_ONNX, HG_ONNX),
                          ("static QDQ int8", FP_Q, HG_Q)]:
    if not (os.path.exists(fp_p) and os.path.exists(hg_p)):
        continue
    size = (os.path.getsize(fp_p) + os.path.getsize(hg_p)) / 1e6
    load_s, rows = bench(fp_p, hg_p)
    print("=" * 76)
    print(f"{label}   |   {size:.1f} MB   |   session load {load_s*1000:.0f} ms "
          f"(once, at startup)")
    print("=" * 76)
    print(f"{'text':32s} {'fastpitch':>10s} {'hifigan':>9s} {'audio':>8s} {'RTF':>7s}")
    for t, f_ms, h_ms, dur, rtf in rows:
        mark = "" if rtf < 0.3 else ("  <-- slow" if rtf < 1 else "  <-- TOO SLOW")
        print(f"{t:32s} {f_ms:8.0f}ms {h_ms:7.0f}ms {dur:7.2f}s {rtf:7.3f}{mark}")
    print()

fp32   |   273.0 MB   |   session load 354 ms (once, at startup)
text                              fastpitch   hifigan    audio     RTF
मदद कीजिए।                             39ms     250ms    0.63s   0.462  <-- slow
गोदाम नंबर 3 पे फायर लग गई है।         98ms     766ms    2.12s   0.407  <-- slow
Sector 21 mein 12 log fase hai        157ms    1315ms    3.69s   0.399  <-- slow

static QDQ int8   |   88.9 MB   |   session load 463 ms (once, at startup)
text                              fastpitch   hifigan    audio     RTF
मदद कीजिए।                             22ms     147ms    0.64s   0.265
गोदाम नंबर 3 पे फायर लग गई है।         62ms     464ms    2.19s   0.240
Sector 21 mein 12 log fase hai        104ms     810ms    3.80s   0.241



## 11. Listen, then package

The only quality check that means anything is a human ear. Play both and confirm
the int8 version is still clearly understandable. If it sounds muffled, buzzy or
clipped, the calibration set is the first thing to change — add more sentences
covering the real vocabulary.

In [23]:
# helpers, in case an earlier cell was skipped
if "onnx_synth" not in globals():
    def onnx_synth(text, length_scale=None, fp=None, hg=None):
        ids = tokenizer.text_to_ids(normalise(text))
        mel = fp.run(None, {
            "tokens": np.array([ids], np.int64),
            "speaker_id": np.array([SPEAKER_ID], np.int64),
            "length_scale": np.array([length_scale or LENGTH_SCALE],
                                     np.float32)})[0]
        return np.asarray(hg.run(None, {"mel": mel})[0]).squeeze(), mel.shape[-1]

sess = {}
for label, fp_p, hg_p in [("fp32", FP_ONNX, HG_ONNX), ("int8", FP_Q, HG_Q)]:
    if fp_p and hg_p and os.path.exists(fp_p) and os.path.exists(hg_p):
        sess[label] = (ort.InferenceSession(fp_p, providers=["CPUExecutionProvider"]),
                       ort.InferenceSession(hg_p, providers=["CPUExecutionProvider"]))

for text in BENCH:
    print("=" * 60)
    print(f"{text}\n  -> {normalise(text)}")
    for label, (a, b) in sess.items():
        w, _ = onnx_synth(text, LENGTH_SCALE, fp=a, hg=b)
        w = trim_and_collapse(w, sr)
        p = f"{AUDIO_DIR}/{label}_{abs(hash(text)) % 9999}.wav"
        sf.write(p, w.astype(np.float32), sr)
        print(f"\n  {label}  ({len(w)/sr:.2f} s)")
        display(Audio(p))

# Same mel to BOTH vocoders. Comparing full pipelines would mix quantisation
# damage with duration drift and inflate the number for no reason.
if len(sess) == 2:
    fp32_fp, fp32_hg = sess["fp32"]
    _,       int8_hg = sess["int8"]
    ids = tokenizer.text_to_ids(normalise(TEST_TEXT))
    mel = fp32_fp.run(None, {
        "tokens": np.array([ids], np.int64),
        "speaker_id": np.array([SPEAKER_ID], np.int64),
        "length_scale": np.array([LENGTH_SCALE], np.float32)})[0]
    a32 = np.asarray(fp32_hg.run(None, {"mel": mel})[0]).squeeze()
    a8  = np.asarray(int8_hg.run(None, {"mel": mel})[0]).squeeze()
    n = min(len(a32), len(a8))
    S1 = np.log(np.abs(librosa.stft(a32[:n].astype(np.float32))) + 1e-8)
    S2 = np.log(np.abs(librosa.stft(a8[:n].astype(np.float32))) + 1e-8)
    print(f"\nvocoder-only log-spectrum L1: {np.abs(S1 - S2).mean():.3f}")
    print("  < 0.5 inaudible | < 1.5 probably fine | > 1.5 suspect calibration")

मदद कीजिए।
  -> मदद कीजिए

  fp32  (0.67 s)



  int8  (0.68 s)


गोदाम नंबर 3 पे फायर लग गई है।
  -> गोदाम नंबर तीन पे फायर लग गई है

  fp32  (2.16 s)



  int8  (2.23 s)


Sector 21 mein 12 log fase hain, ambulance turant bhejo.
  -> सेक्टर इक्कीस मेइन् बारह लोग् फ़से हैन्, एम्बुलेंस तुरन्त् भेजो.

  fp32  (3.69 s)



  int8  (3.84 s)



vocoder-only log-spectrum L1: 1.655
  < 0.5 inaudible | < 1.5 probably fine | > 1.5 suspect calibration


In [24]:
import shutil
from IPython.display import FileLink

# ship the int8 pair plus everything Android needs to reproduce tokenisation
meta = {
    "lang": LANG,
    "sample_rate": sr,
    "hop_length": HOP,
    "num_mels": N_MELS,
    "speaker_id": int(SPEAKER_ID),
    "speaker_name": SPEAKER_NAME,
    "length_scale_default": LENGTH_SCALE,
    "max_chunk_chars": MAX_CHUNK_CHARS,
    "chunk_gap_ms": 60,
    "needs_transpose": bool(NEEDS_TRANSPOSE),
    "vocab": list(VOCAB),
    "inputs": {"fastpitch": ["tokens", "speaker_id", "length_scale"],
               "hifigan": ["mel"]},
}
json.dump(meta, open(f"{ONNX_DIR}/meta.json", "w"), ensure_ascii=False, indent=2)

with open(f"{ONNX_DIR}/tokens.txt", "w", encoding="utf-8") as f:
    for i, c in enumerate(VOCAB):
        f.write(f"{c} {i}\n")

zp = shutil.make_archive(f"{WORK_DIR}/onnx_{LANG}", "zip", ONNX_DIR)
print(f"{zp}   {os.path.getsize(zp)/1e6:.1f} MB")
print("\ncontents:")
for f in sorted(os.listdir(ONNX_DIR)):
    print(f"   {f}  ({os.path.getsize(os.path.join(ONNX_DIR, f))/1e6:.1f} MB)")
display(FileLink(zp))

/kaggle/working/onnx_en+hi.zip   422.1 MB

contents:
   fastpitch_en+hi.onnx  (217.2 MB)
   fastpitch_en+hi.qdq-safe.onnx  (66.6 MB)
   fastpitch_en+hi.static-qdq.onnx  (63.1 MB)
   hifigan_en+hi.minmax.onnx  (22.3 MB)
   hifigan_en+hi.onnx  (55.7 MB)
   hifigan_en+hi.qdq-safe.onnx  (22.3 MB)
   hifigan_en+hi.static-qdq.onnx  (14.3 MB)
   meta.json  (0.0 MB)
   tokens.txt  (0.0 MB)


/kaggle/working/onnx_en+hi.zip

## 12. Playground — type anything and hear it

Edit `MY_TEXT`, run the cell. It normalises, splits, synthesises through
whichever ONNX pair you settled on, and prints the timings.

Write the way your users will: mixed script, numbers as digits, no careful
punctuation. That is what the model has to survive.

In [28]:
# ══════════════════════════════════════════════════════════════════════
MY_TEXT = "Maulik kuch kaam nahi karta"
USE_INT8 = True          # False to hear the fp32 pair
# ══════════════════════════════════════════════════════════════════════

_fp, _hg = (s_int8_fp, s_int8_hg) if (USE_INT8 and s_int8_fp and s_int8_hg) \
           else (s_fp32, s_hg32)
_tag = "int8" if (USE_INT8 and s_int8_fp) else "fp32"

norm = normalise(MY_TEXT)
chunks = split_clauses(norm)

print("=" * 72)
print(f"  raw    : {MY_TEXT}")
print(f"  norm   : {norm}")
print(f"  tokens : {len(tokenizer.text_to_ids(norm))}")
print(f"  chunks : {len(chunks)}")
for c in chunks:
    print(f"           - {c}")
print(f"  model  : {_tag},  length_scale {LENGTH_SCALE}")
print("=" * 72)

dropped = [c for c in norm if c not in VOCAB_SET and not c.isspace()]
if dropped:
    print(f"\n  WARNING: dropped by tokenizer: {sorted(set(dropped))}")
    print("  Add these to LATIN_TO_DEVA or accept they will be silent.\n")

w, timings = synth_long(MY_TEXT, fp=_fp, hg=_hg)
if len(w) == 0:
    print("\n  nothing survived normalisation - check the warning above")
else:
    total = len(w) / sr
    p = f"{AUDIO_DIR}/playground.wav"
    sf.write(p, w, sr)
    print(f"\n  audio          {total:.2f} s")
    print(f"  synthesis      {sum(timings):.2f} s   RTF {sum(timings)/total:.3f}"
          f"{'   <- under 0.3, good' if sum(timings)/total < 0.3 else ''}")
    print(f"  time to first  {timings[0]*1000:.0f} ms")
    display(Audio(p))

print("""
Change MY_TEXT and re-run. Worth trying:
  - a very long sentence, to confirm chunking holds
  - pure Devanagari vs pure Latin vs mixed
  - numbers written as digits: "23 log", "sector 45", "raat 8 baje"
  - a name the model has never seen
""")

  raw    : Maulik kuch kaam nahi karta
  norm   : मौलिक् कुच् काम् नहि कर्त
  tokens : 25
  chunks : 1
           - मौलिक् कुच् काम् नहि कर्त
  model  : int8,  length_scale 0.9
   [1/1]  25 tok ->  1.60s in   394ms   मौलिक् कुच् काम् नहि कर्त

  audio          1.60 s
  synthesis      0.39 s   RTF 0.246   <- under 0.3, good
  time to first  394 ms



Change MY_TEXT and re-run. Worth trying:
  - a very long sentence, to confirm chunking holds
  - pure Devanagari vs pure Latin vs mixed
  - numbers written as digits: "23 log", "sector 45", "raat 8 baje"
  - a name the model has never seen



## What you have, and what is still open

If section 9 said PASS and the int8 audio is understandable, you are holding an
ONNX export of Indic-TTS `en+hi` with runtime speed control. `meta.json` and
`tokens.txt` are in the zip so the Android side can reproduce tokenisation
exactly.

**Re-run with a different `LANG`** to do `gu`, `or`, `hi` or `en`. Nothing above
is language-specific except `CALIB_TEXTS`, `LATIN_TO_DEVA` and `hi_num` — and
those three are exactly the work each new language needs.

### Still open

- **ARM.** `ConvInteger = 0` guarantees the graph loads on a phone. The RTF here
  is from a Kaggle CPU and must be re-measured on the device.
- **Two-stage handoff.** FastPitch and HiFi-GAN are two sessions, so there is a
  cost that single-file VITS models do not pay. Compare fairly per language
  before choosing.
- **Number-to-words for the other nine languages.** `hi_num` is the template.
  Each language is a few hundred lines and every one is audible in a demo.
- **Clause splitting.** Your ASR output has no punctuation, because CTC does not
  produce any. So splitting on `.`/`,`/`।` for chunked playback will never fire
  on Indic text. The fix belongs on the sending phone: VAD already knows where
  the speaker paused, so send those pause positions as byte offsets in the
  packet. Costs a few bytes, needs no extra model, and reproduces the speaker's
  real prosody better than a punctuation model would.
- **Quality is still ear-only.** MOS with five native raters per language is the
  real bar.